# <center>Fufan-CC Flow 项目：CLI 集成进 Web 的范式拆解</center>

&emsp;&emsp;今天我们聚焦的是一个很多 AI 应用工程师没碰过、但近一年突然爆发的工程范式——**把命令行 AI Agent 集成进 Web**。具体到本课的样本，就是把 `Claude Code CLI` 这种本地终端工具，通过 `Claude Agent SDK` + `node-pty` + `WebSocket` 包成一个浏览器里能用的图形化前端。`Fufan-CC Flow` 就是这样一个开源项目，它把 Claude Code 的全部能力——实时对话流、工具调用可视化、HIL 权限确认、Session 管理、上下文压缩、终端集成——都做进了 Web UI。

&emsp;&emsp;为什么现在值得学？因为如果你之前做的是 `RAG` 或 `LangChain Agent`，你的整条工作流是"应用直连大模型 API"——`OpenAI` / `Claude` / `DeepSeek` 一个 HTTP 请求过去拿回答。但 `CLI 集成进 Web` 是另一条工作流：**应用层不直接调模型，而是包装一个本地 CLI 进程**。这条路解决的问题完全不同——它要解决"如何把已经设计好的 CLI 智能体（带工具调用、带文件系统访问、带 MCP）搬到浏览器里给用户"。`Cursor` / `aider` / `goose` / `Fufan-CC` 都在走这条路。

&emsp;&emsp;结合本课的真实代码与现场调研，我们会重点抓住三件事：一是 `Fufan-CC` 的**双泳道架构**（`Chat/Agent 泳道` 走 `Agent SDK`，`Terminal 泳道` 走 `node-pty`），看清"两类进程为什么必须并存"；二是 **UI 任意元素 ↔ SDK 参数** 的精准对应关系，并量化标注 4 类后端转译（prompt 改写 / 后端调度 / 后端加工 / UI 聚合），破掉"UI 上看到啥 SDK 就有啥"的惯性误判；三是 **零侵入哲学**——`Fufan-CC` 不发明自己的数据存储，直接读 `~/.claude/projects/<hash>/*.jsonl`（CLI 自己落盘的事实源），但在事实源之上做了 4 件加工，把"原始 transcript"翻译成"UI 能渲染的状态"。

&emsp;&emsp;为了把这些内容讲透，我们会按"跑起来 → 看到形状 → 能操作 → 理解边界"的主线展开：先用 15 分钟完成环境部署和项目启动（第一章），再用 20 分钟走读全部功能域（第二章），然后用 20 分钟从浏览器演示倒推出双泳道架构图（第三章），接着用 55 分钟把 UI 上的 10 种交互逐一映射到 Python `claude-agent-sdk` 的具体参数，并用多段真跑 Python 代码复现 SDK 核心机制——从 query 流式消费到 HIL 异步桥接到 WebSocket 整合（第四章），最后用 30 分钟解释"零侵入"背后的设计哲学，给你一份可以套到 `gh CLI` / `aider` / 任何 CLI 上的 5 步迁移骨架（第五章）。

> 📌 **目标受众与前置要求**：本课面向已经熟练掌握 `RAG` / `LangChain` / `Python asyncio` / `WebSocket` 概念的 AI 应用工程师。技术上你需要本机已安装 `Node.js 22.x`、`pnpm 8.x+`、`Python 3.10+` 和 `Claude Code CLI`——第一章会手把手带你完成 `Fufan-CC Flow` 的本地部署，第四章的多段真跑代码需要 Python 环境。

> 📌 **学完本节你将带走 6 件产物**：① 一套在自己机器上跑通的 `Fufan-CC Flow` 完整环境（含 Agent Teams 实验功能）；② 一张能画出来的双泳道架构图（Chat/Agent + Terminal 两条独立进程链）；③ 一张 10 行的 UI ↔ SDK 映射表 + 4 类后端转译分类法；④ 一份能在自己机器跑通的 Python `claude-agent-sdk` 基础对话 + HIL 异步桥接代码；⑤ 对"零侵入读 `~/.claude/projects/`"的具体认知（4 件加工是什么、为什么必要）；⑥ 一份 5 步迁移骨架，能讲给同事听"如果让你把另一个 CLI 包成 Web 应用，第一步该做什么"。

> **【学完不能做·诚实划界】**：本课不会让你能给 `Fufan-CC` 提交 PR（那需要熟悉它的前端 `Zustand` 状态管理 + `Tailwind CSS v4` 设计系统 + 完整测试链）；也不会让你能"包出一个新的 CLI 集成"——5 步迁移骨架是骨架不是完整方案，每一步深入下去都还有大量工程细节。

> 📅 **时效性说明**：本课全部源码引用截止 2026 年 5 月，基于 `Fufan-CC Flow` GitHub 主分支当时的代码状态。所有 `file:line` 引用都是真实可核对的——你可以在自己电脑 `git clone` 仓库后用 `vim server/src/services/claudeAgentService.ts +113` 这样的命令打到对应位置。`claude-agent-sdk` 的 Python 版本依赖也以课件 `requirements.txt` 锁定的版本为准。（第四章开头会用一段 `pip install -r requirements.txt` 一次性把 Python 依赖装齐，目前不需要提前安装）

---

## <center>第一章：环境部署与项目启动</center>

&emsp;&emsp;本章的目标只有一件事：让你在自己的机器上把 `Fufan-CC Flow` 完整跑起来，并完成从环境准备、服务启动，到发出第一条对话的全流程验证。这不是走过场的"能跑就行"——我们会在每一步都核实服务状态，确保你进入下一章时面对的是一个完全就位的开发环境。

&emsp;&emsp;整个过程分三段递进：**1.1 环境准备**帮你在动手之前就把隐患清零；**1.2 安装启动**走完 `pnpm install → pnpm dev`，验证 3 栏首屏；**1.3 配置闭环**完成 `Settings` 四件套，打通到第一条消息。按顺序走下来，整个过程约 15 分钟。

### 1.1 环境准备：依赖清单与版本约束

&emsp;&emsp;`Fufan-CC Flow` 是一个 `Monorepo` 项目，前后端共存于同一仓库，通过根目录的 `pnpm-workspace.yaml` 把 `client/` 和 `server/` 串在一起统一管理。这种结构带来一个必须提前知道的硬约束：`Node.js` 的版本不能随意选——后端用到了 `node-pty` 来驱动真实终端，而 `node-pty 1.1` 的预编译二进制只覆盖到 `Node 22`，`Node 26` 会直接编译失败。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Fufan-CC Flow 环境依赖清单</font></p>
<div class="center">

| 依赖 | 版本要求 | 说明 |
|------|----------|------|
| `Node.js` | **22.x LTS** | Node 26 与 `node-pty` 1.1 预编译不兼容（已实测） |
| `pnpm` | 8.x+ | Monorepo 工作区，根 `pnpm-workspace.yaml` 串前后端 |
| `Claude Code CLI` | 最新 | macOS/Linux: `curl -fsSL https://claude.ai/install.sh \| bash` / Windows: PowerShell `npm install -g @anthropic-ai/claude-code`，必须本机已登录或有 API Key |
| Python + C++ 工具链 | — | macOS `xcode-select --install` / **Windows: Node.js .msi 安装时勾选 "Automatically install the necessary tools"**（一次性装 Python+VS Build Tools） / Linux `build-essential` |
| **Git for Windows**（Windows 独有） | 最新 | Claude Code CLI 在 Windows 上需要 Git Bash；fufan-cc-flow 自动检测 `C:\Program Files\Git\bin\bash.exe` 等 4 个候选路径，可用项目根 `.env` 文件的 `CLAUDE_CODE_GIT_BASH_PATH` 覆盖 |

</div>

&emsp;&emsp;**Windows 学员快速通道**（与 macOS/Linux 主线并行，基于项目 README.md 与源码实测）：

&emsp;&emsp;**第一步**，下载 [Node.js 22.x LTS .msi](https://nodejs.org/zh-cn) 安装包，安装时勾选 **"Automatically install the necessary tools"**——一次性把 Python 和 VS Build Tools 装齐，避免后面 `pnpm install` 时 `node-pty` 编译失败。

&emsp;&emsp;**第二步**，从 [git-scm.com](https://git-scm.com/download/win) 装 Git for Windows（含 Git Bash），选项选 "Git from the command line and also from 3rd-party software"。Git Bash 是 Claude Code CLI 内部跑工具调用时必需的 shell host。

&emsp;&emsp;**第三步**，在 PowerShell 中跑 `npm install -g pnpm @anthropic-ai/claude-code`。如遇权限错误，先执行 `Set-ExecutionPolicy -Scope CurrentUser -ExecutionPolicy RemoteSigned`，或以管理员身份重开 PowerShell。

&emsp;&emsp;**第四步**，PowerShell 或 Git Bash 任一都可：`git clone <repo-url> fufan-cc-flow-src && cd fufan-cc-flow-src && pnpm install`。如果 `node-pty` 编译失败，手动装 [VS Build Tools 2022](https://visualstudio.microsoft.com/visual-cpp-build-tools/) 勾选 **"C++ 桌面开发"** 工作负载即可修复。

&emsp;&emsp;**第五步**，跑 `pnpm dev` 启动前后端，浏览器打开 `http://localhost:5173`。如果 fufan-cc-flow 检测不到 Git Bash 路径，在项目根目录新建 `.env` 文件加一行 `CLAUDE_CODE_GIT_BASH_PATH=C:\你的\Git\路径\bin\bash.exe`。

> **【源码锚点】**：`server/src/services/ptyService.ts:39-57`（Windows 终端泳道走 `cmd.exe` + `useConpty: false` 用 winpty 后端，避开 ConPTY 的 `STATUS_CONTROL_C_EXIT 0xC000013A` 错误）/ `systemService.ts:64-71`（Git Bash 4 候选路径检测）/ `files.ts:53-65`（Windows 文件浏览器枚举 C-Z 盘符）/ `ClaudeEnvPanel.tsx:211-307`（前端 UI 检测到 `platform === "win32"` 切 Windows 提示分支）。

&emsp;&emsp;装好之后，先用两条命令做一次快速自检，确认两个关键二进制都能正常调用：

```bash
node -v          # ⇒ v22.x
claude --version # ⇒ 必须能出版本号，否则后续 SDK 起不来
```

**Agent Teams 环境变量**

&emsp;&emsp;这一步很多人会跳过，等到 Phase 7 再撞墙。我们在这里一次性把它做掉。`Fufan-CC Flow` 右栏的 `Teams Tab` 默认会显示"`Agent Teams` 未启用"，根因是后端的 `teamService.isEnabled()` 只读 `process.env`，<font color=red>不读</font> `~/.claude/settings.json`——那个配置文件只给 `claude` CLI 自身用，`Express` 后端进程完全访问不到它。

&emsp;&emsp;**macOS / Linux**（zsh / bash）：

```bash
echo 'export CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS=1' >> ~/.zshrc
source ~/.zshrc
```

&emsp;&emsp;**Windows**（PowerShell，用户级持久化）：

```powershell
[Environment]::SetEnvironmentVariable("CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS", "1", "User")
# 写入后必须关闭当前 PowerShell 窗口、重开新窗口，让用户级环境变量生效
```

&emsp;&emsp;写入之后，用下面这条命令验证变量已在当前 shell 生效：

```bash
# macOS / Linux：
echo $CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS  # ⇒ 应输出 1

# Windows PowerShell：
# echo $env:CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS  # ⇒ 应输出 1
```

> **【踩坑预警】**：必须在能 `echo` 出该变量的 <font color=red>同一个 terminal</font> 里执行 `pnpm dev`——Node 进程的 env 是启动那一刻的快照，settings.json 写一万遍也不会被 fufan-cc-flow 后端读到。Windows 上 `[Environment]::SetEnvironmentVariable(..., "User")` 必须重开 PowerShell 窗口才能生效，不能在同一窗口内"立刻 `pnpm dev`"。

> **【源码锚点】**：`server/src/services/teamService.ts:46`（判定逻辑就一行 `process.env.CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS === "1"`）

### 1.2 安装启动：从 pnpm install 到 3 栏首屏

&emsp;&emsp;环境就位之后，安装过程本身很快——`pnpm` 的工作区机制会一次性把前后端所有依赖都装好，`node-pty` 也会在这步自动完成本机编译。装完之后，`pnpm dev` 会用 `concurrently` 同时拉起前后端两个服务，我们需要确认两个端口都正常监听。

```bash
cd fufan-cc-flow-src && pnpm install && pnpm dev
```

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Fufan-CC Flow 服务端口</font></p>
<div class="center">

| 端口 | 服务 | 入口源码 |
|------|------|----------|
| `:3001` | Express + WS | `server/src/index.ts` |
| `:5173` | Vite 前端 | `client/src/main.tsx → AppLayout` |

</div>

&emsp;&emsp;两个端口都起来之后，打开浏览器访问 `http://localhost:5173`。能看到完整的 3 栏 UI——左栏 `Sidebar`、中栏 `ChatPanel`、右栏 `RightPanel` 都正常渲染——即代表本节通过。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140042845.png" width=50%></div>

&emsp;&emsp;（首屏 3 栏布局——左栏 Sidebar、中栏 ChatPanel、右栏 RightPanel）

> **【源码锚点】**：`client/src/components/layout/AppLayout.tsx:14`

### 1.3 配置闭环：从 Settings 到第一条对话

&emsp;&emsp;看到首屏不代表可以直接聊天——中栏输入框此时还是灰色不可用状态。我们需要完成 `Settings` 四件套配置，把运行时所需的环境检测、认证凭据、模型选择和工作目录全部就位。操作入口是左下角的 **Settings 按钮**，点击后进入 `SettingsModal`。

**步骤一：环境检测**

&emsp;&emsp;Settings → Step 1，系统会自动检测 `Node.js` 版本、`Claude Code CLI` 是否可调用、网络连通性这三项状态。全部绿灯之后再往下走。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140052187.png" width=50%></div>

**步骤二：API Key 或 OAuth 授权**

&emsp;&emsp;Settings → Step 2，根据你的使用方式选择填入 `apiKey` 和 `baseURL`（使用国产基座时需要填 `baseURL`），或者通过 OAuth 完成授权。

> **【踩坑预警】**：在 macOS 上，`Claude Code CLI` 自身把 OAuth 凭据存入系统 **Keychain**（服务名 `Claude Code-credentials`），而 fufan-cc-flow 的鉴权状态检测仅 `existsSync(~/.claude/.credentials.json)` —— 因此 UI 会显示"未授权"，但这是已知的 UI 误报：WebSocket 发送链路本身不做鉴权 gate，CLI 子进程会自行从 Keychain 取凭据完成请求，<font color=red>不影响实际发消息</font>。验证方式很简单：直接在中栏发一条消息，能收到回复就代表真的授权了。

> **【源码锚点】**：`server/src/services/claudeSettingsService.ts:54-56`（`hasOAuthCredentials()` 仅 `existsSync(.credentials.json)`，未查 Keychain）+ `systemService.ts:148-159`（`getAuthStatus()` 调用层）。

> **【Windows 学员注意】**：上述 Keychain 误报机制**仅适用于 macOS**。Windows 上 fufan-cc-flow 同样调 `existsSync` 检查文件（路径在 Windows 上解析为 `C:\Users\<username>\.claude\.credentials.json`），但 Claude Code CLI 在 Windows 上的实际凭据存储位置由 CLI 自身决定（本课范围不展开）。如果 Windows 上 UI 显示"未授权"，参照与 macOS 同样的"直接发消息验证"方法判断真实状态。

**步骤三：模型选择**

&emsp;&emsp;Settings → Model，可选内置的 `sonnet` / `opus` / `haiku`，也可以填入自定义模型名（对接国产基座时常用）。选好后配置会持久化到 `configStore`，后续会话都会沿用。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140052218.png" width=50%></div>

> **【源码锚点】**：`client/src/components/manage/ModelSelector.tsx`

**步骤四：选择工作目录**

&emsp;&emsp;回到主界面，点击左栏顶部的"点击选择项目文件夹"，在 `FolderBrowserModal` 里选择你要让 Claude 操作的代码目录。这一步决定了后续工具调用的根路径，选一个真实的代码项目效果最好。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140043018.png" width=50%></div>

> **【源码锚点】**：`FolderBrowserModal`

&emsp;&emsp;四步全部完成之后，中栏输入框就会变为可用状态。这意味着我们已经准备好发出第一条消息了——在 2.2 节我们会立刻验证这条通路，同时观察流式 token、Thinking 折叠块、右栏 Token 计数和累计花费这四个产物是否同时出现。

&emsp;&emsp;<font color=red>**第一章到这里完成**</font>——你已经把 `Fufan-CC Flow` 完整跑在本机上、把 `Settings` 四件套配齐、把第一条消息的通路打通。接下来每一章都是在这个"已经能跑通"的基础上往下挖：第二章带你走完所有功能域建立感性认知，第三章把 UI 上看到的东西反推回架构图，第四章把每个 UI 交互精确映射到 SDK 参数并用 Python 复现，第五章退到一万米高度看零侵入设计哲学和迁移骨架。

---

## <center>第二章：功能全景——从首条对话到 Agent 协作</center>

&emsp;&emsp;上一章我们完成了环境就位和项目启动，`pnpm dev` 已经能跑起来、首屏 3 栏 UI 也确认可见了。本章的目标是用大约 20 分钟，把 Fufan-CC 的全部功能域走读一遍——让你在进入第 3 章做架构拆解之前，先对"这个项目能做什么"建立完整的感性认知。

&emsp;&emsp;我们会按使用频率从高到低展开：2.1 先给你一张全局快照，建立空间感；然后依次走读首条对话验证（2.2）、工具调用与 `HIL` 权限（2.3）、IDE 集成能力（2.4）、拓展系统（2.5），最后是最复杂的 Agent 系统与会话管理（2.6）。每一节都是独立可操作的验证点，不是抽象描述。

### 2.1 项目快照：3 栏布局与 6 个全局弹窗

&emsp;&emsp;Fufan-CC 的 UI 是经典的 IDE 三栏布局——左侧文件与工具面板、中间主对话区、右侧监控与扩展区。理解每个栏的职责，是后续所有功能讲解的空间前提。先看一眼带标注的全局截图：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140033160.png" width=50%></div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Fufan-CC Flow UI 区域与功能分布</font></p>
<div class="center">

| 区域 | 核心功能 | 入口源码 |
|------|----------|----------|
| 左栏 Sidebar | 3 面板（Files / Search / Checkpoints）+ Settings + ContextBar | `client/src/components/layout/Sidebar.tsx` |
| 中栏 ChatPanel | 流式对话 / 工具卡片 / HIL 弹窗 / 附件 / Slash 命令 | `client/src/components/chat/ChatPanel.tsx` |
| 右栏 RightPanel | 3 一级 Tab（Monitor / 拓展 / Agent）+ 终端 Tab | `client/src/components/layout/RightPanel.tsx` |
| 全局 Modal | History / FileView / Settings / FolderBrowser / SkillBrowser / CreateSkill（共 6 个）| `client/src/components/modals/` |
| 后端 REST | agents / files / mcp / memory / sessions / system / teams / workflows 等 15 路 | `server/src/routes/` |
| 后端 WS | chatHandler（对话流）/ terminalHandler（终端 I/O）| `server/src/websocket/` |

</div>

&emsp;&emsp;把数字感建立一下：3 栏 + 6 个全局 `Modal` + 15 路 `REST` + 2 路 `WebSocket`。对于一个前后端一体的 AI IDE，这个规模不算大——代码库结构非常清晰，每一块都有对应的单一职责文件。这也是我们第 3 章能快速拆解架构的前提。

### 2.2 首条对话与流式输出验证

&emsp;&emsp;项目配置完成后，第一件事永远是发一条最简单的消息，验证对话链路是否完整通畅。这是最小验证闭环——能跑通这一步，说明从前端 WebSocket、后端 `chatHandler`、`claudeAgentService`，一直到 Claude Code CLI 子进程的整条链路都没问题。

&emsp;&emsp;在中栏输入框输入"你好"，回车。正常情况下，4 个可观测产物会几乎同时出现：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>首条对话的 4 个可观测产物</font></p>
<div class="center">

| 产物 | 渲染位置 | 源码 |
|------|----------|------|
| 流式 token | `MessageList.tsx` 气泡 | `chat/MessageList.tsx` |
| Thinking 折叠块 | 气泡内可展开 | 同上，`thinking` 段类型 |
| 实时 Token 计数 | 右栏 Live Monitor | `RightPanel.tsx:543` SummaryItem |
| 累计花费 `$` | 右栏 Live Monitor | 同上 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140042034.png" width=50%></div>

&emsp;&emsp;把通信链路抽象出来看，这条消息的背后涉及多跳跨层调用——浏览器输入栏经过 WebSocket 通道进入后端 Node 服务，再 spawn 出 CLI 子进程消费流式输出，最后把消息推回浏览器渲染。这条链路的完整 8 步分解会在第三章 3.3 节用一张专门的飞行路径图给出，本节只需要建立"通信是流式的、跨多层"的整体感觉。

> **【踩坑预警】**：如果发消息后超过 60 秒才有响应，先看 `pnpm dev` 后端日志里的 MCP server 启动数量。每个 MCP 都是独立的 subprocess，<font color=red>配置了超过 10 个 MCP 时，冷启动阶段会显著拉慢首次响应</font>。

### 2.3 工具调用与 HIL 权限确认

&emsp;&emsp;对话链路跑通之后，下一步是验证 `Fufan-CC` 与普通聊天 UI 最大的差异——工具调用可视化和 `HIL`（Human-in-the-Loop）权限管理。这不只是 UI 展示，它背后有一套值得深入理解的异步工程机制。

&emsp;&emsp;发一条会触发工具的消息，比如"列出当前目录所有文件"。对话区会出现 3 类产物：`ToolCallCard` 展示 `Bash` / `Read` 等工具的输入与折叠结果；对于 `rm` / `write` 等危险操作，`HIL` 弹窗会弹出权限确认 Modal，阻塞执行直到人工确认；工具结束后出现 `TaskResultCard` 小结卡。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140052868.png" width=50%></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140044287.png" width=70%></div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>HIL 权限确认的 3 种策略</font></p>
<div class="center">

| 策略 | 行为 |
|------|------|
| 一次性批准 | 仅本次允许，下次同类操作仍弹窗 |
| 永久批准 | 写入项目级 settings，后续不再弹窗 |
| 拒绝 | 工具返回错误给 SDK，对话继续 |

</div>

&emsp;&emsp;<font color=red>`HIL` 是这个项目最值得深入的工程亮点，值得专门说一句：SDK 给的 `canUseTool` 回调（TS 端 camelCase；对应 Python SDK 的 `can_use_tool`）虽然本身是 `async` 签名，但语义上要求"调用 → 拿到 PermissionResult → 工具才继续执行"的阻塞流程；而前端用户决策走的是 WebSocket 推弹窗 + 用户点击的真正异步路径——中间用 Promise + Map 做了一个异步桥接。这个机制我们在第 4.4 节会用 Python 最小版复现，届时你会看到这层桥接的精妙之处。</font>

> **【源码锚点】**：`server/src/services/claudeAgentService.ts:108-180` 的 `canUseTool` 实现（TS 端命名空间——对应 Python SDK 同一字段名为 `can_use_tool` snake_case，两者是 SDK 跨语言提供的同一回调钩子的两种命名形式）——这是异步桥接的核心所在；本课 4.4 节用 Python `can_use_tool` 复现同一机制。

### 2.4 IDE 集成：文件树、终端与代码查看

&emsp;&emsp;Fufan-CC 不只是一个聊天窗口，它还集成了完整的 IDE 能力。理解这一点很重要：当 Claude 在修改你的代码文件时，你不需要切换到外部编辑器——文件浏览、代码差异查看、真实终端，全部都在同一个界面里。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Fufan-CC IDE 集成功能一览</font></p>
<div class="center">

| 入口 | 功能 | 源码 |
|------|------|------|
| 左栏 Files | 项目文件树，点击在 `FileViewModal` 打开 | `components/ide/FileTree.tsx` |
| 左栏 Search | 全文件名搜索（注意：不是内容搜索）| `Sidebar.tsx:241` SearchPanel |
| 左栏 Checkpoints | Claude 修改文件后的快照时间线，可回滚 | `components/agent/CheckpointTimeline.tsx` |
| 右栏终端 Tab | xterm.js + node-pty 真 Shell，点 `+` 号添加多个 | `components/ide/Terminal.tsx` + `ptyService.ts` |
| 代码查看器 | `CodeMirror 6` 语法高亮 | `components/ide/CodeViewer.tsx` |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140039961.png" width=50%></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140052334.png" width=50%></div>

&emsp;&emsp;需要特别强调的是：右栏终端面板和中栏的聊天面板是**完全独立的两条链路**。终端走 `terminalHandler.ts → ptyService.ts`，对话走 `chatHandler.ts → claudeAgentService.ts`，两者互不干扰。这正是第 3 章要讲的"双泳道架构"的直接体现。

> **【踩坑预警】**：终端报 <font color=red>"PTY spawn failed"</font>，去看 `server/src/websocket/terminalHandler.ts:18` 的 try/catch——通常是 `node-pty` 没有本机编译，执行 <font color=red>`pnpm rebuild node-pty`</font> 可以修复。

### 2.5 拓展系统：MCP / Skills / Memory / Hooks

&emsp;&emsp;右栏"拓展"Tab 集中了 Fufan-CC 的可扩展能力。这里的每一个子面板，都与 Claude Code CLI 的某个原生能力对应——如果你熟悉 CLI 的配置方式，可以直接把这里理解成一个图形化的配置管理界面。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140039217.png" width=50%></div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>拓展 Tab 5 子面板与 CLI 等价功能</font></p>
<div class="center">

| 子 Tab | CLI 等价 | 能做的事 |
|--------|----------|----------|
| MCP | `claude mcp add` | 图形化添加 stdio / HTTP / SSE 类型 MCP Server |
| Skills | `~/.claude/skills/` + `/skill-name` | 浏览 / 创建 / 启用 Skill，触发 SkillBrowserModal |
| 插件 | Marketplace plugins | 插件市场 + 已装插件管理 |
| Memory | `/memory` + `CLAUDE.md` | 双体系：Auto Memory（项目级 memory/）+ 项目级 CLAUDE.md |
| Hooks | `~/.claude/settings.json hooks` | PreToolUse / PostToolUse 等生命周期钩子 |

</div>

&emsp;&emsp;以最常用的 MCP Manager 为例，下面这张截图展示了它的典型界面——左侧是已装 MCP 列表，右侧是配置详情和启停控制，本质上就是给 `~/.claude.json` 的 `mcpServers` 字段做了一个图形化包装：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140049561.png" width=50%></div>

> **【源码锚点】**：`components/manage/` 各 Manager 组件 + `server/src/services/` 同名 service——前后端命名一一对应，阅读成本很低。

### 2.6 Agent 系统与会话管理

&emsp;&emsp;右栏"Agent"Tab 是项目最复杂、也是最有价值的一块。它不只是 UI 展示——它把 Claude Code 原生的 sub-agent 调度、后台任务、多 agent 协作能力，全部用图形界面暴露了出来。我们把 Agent 系统和会话生命周期放在同一节讲，是因为两者共同构成了"单次对话之外"的完整工作流。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Agent Tab 5 子面板功能</font></p>
<div class="center">

| 子 Tab | 功能 | 源码 |
|--------|------|------|
| Agent 管理 | 查看 / 创建 sub-agent（对应 `~/.claude/agents/`）| `components/agent/AgentManager.tsx` |
| 执行树 | 当前会话的 sub-agent 调用链可视化 | `SubAgentTree.tsx` |
| 后台任务 | 长跑任务面板（dispatch 后不阻塞主对话）| `BackgroundTasks.tsx` + `TaskBoard.tsx` |
| 工作流 | Workflow 编排（多 agent 串联）| `WorkflowManager.tsx` |
| Teams | Agent Teams（CC 原生 multi-agent 协作）——需 1.1 节已 export 环境变量 | `TeamPanel.tsx` + `TeamCreator.tsx` |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140033183.png" width=50%></div>

**会话生命周期**

&emsp;&emsp;除了 Agent 系统，另一块经常被忽略的能力是会话的生命周期管理。点击顶栏 **History 按钮**，`HistoryModal` 会列出 `~/.claude/projects/<hash>/` 下所有历史 session；点击某条历史记录选择 **Resume**，可以复用 `session_id` 继续上一次的对话，上下文完全连续。当对话内容超出 context window 时，中栏会出现 `CompactDivider`，可视化地标记出上下文压缩事件的位置。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140046018.png" width=50%></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140038749.png" width=50%></div>

> **【源码锚点】**：`components/modals/HistoryModal.tsx` + `server/src/services/sessionManager.ts`

**常见故障速查表**

&emsp;&emsp;在走完全部功能之前，把下面这张故障速查表过一遍——这 6 条覆盖了新人最高频遇到的问题，每条都有精确的源码锚点，排查起来不需要到处找。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>常见故障速查表</font></p>
<div class="center">

| 症状 | 诊断锚点 | 通常根因 |
|------|----------|----------|
| `Internal Server Error` 首屏 | 后端 `pnpm dev` 日志 | CLAUDE.md 解析报错 / node-pty 未编译 |
| 发消息报 `executable not found at cli.js` | `claudeAgentService.ts:65` | env 没 spread `process.env`，子进程丢 PATH |
| 终端 `posix_spawnp failed` | `terminalHandler.ts:18` | node-pty 没本机编译，重跑 `pnpm rebuild node-pty` |
| UI 显示"未授权"但实际能用 | `claudeSettingsService.hasOAuthCredentials()` | UI 检测仅查 `.credentials.json` 文件，未覆盖 macOS Keychain 存储路径（误报，不影响使用）|
| 单条响应 >60s | 后端启动日志 MCP 数 | MCP 太多，每条 query 都重启全部 server |
| Teams Tab 显示"未启用" | `teamService.ts:46` | shell 没 export 环境变量；写 `settings.json` 对此后端无效 |

</div>

&emsp;&emsp;到这里，你已经完整走过了 Fufan-CC 的全部功能域——从最基础的首条对话验证，到工具调用可视化、IDE 集成、拓展系统，一直到 Agent 协作和会话管理。接下来我们从"用"转向"理解"：第 3 章会把你刚才看到的这些 UI 交互，一层一层还原成内部架构，让你不只是会用，还能讲得清楚。

---

## <center>第三章：架构拆解</center>

&emsp;&emsp;前两章我们完成了部署并走读了全部功能。从这里开始，我们从”用”转向”理解”——把你在浏览器里看到的那些 UI 交互，还原成内部的工程结构。本章的目标是让你在脑里建立起一个能容纳后两章所有内容的**整体架构图**——后面的 UI ↔ SDK 映射、Python 复现、零侵入哲学，都会回指这张图里的某一层、某一个步骤。

&emsp;&emsp;我们会先用一张表把项目的核心技术栈摊在桌上（看清"用了什么"），再按三个递进的小节展开拆解（看清"怎么组织"）：先用一张全局鸟瞰图把项目的 6 层分层和 3 条通信通道摊开（3.1）；再聚焦到最核心的两条通道——`Chat/Agent` 泳道和 `Terminal` 泳道——讲清楚双泳道 5 层架构（3.2）；最后用一条”单条消息飞行路径”把静态架构图变成动态视图，让你看清一次发送在 5 层之间是怎么走的（3.3）。3.3 末尾的 8 步飞行路径是整门课最重要的锚点，后两章所有讲解都会回引到它的某一步。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Fufan-CC Flow 核心技术栈</font></p>
<div class="center">

| 端 | 技术 | 版本 | 作用 |
|---|---|---|---|
| **前端** | `React` | `19` | 3 栏 UI + 6 个 `Modal` 渲染 |
| 前端 | `Vite` | `6` | 开发服务器（`:5173`）+ 生产构建 |
| 前端 | `Tailwind CSS` | `4` | 原子化样式系统 |
| 前端 | `Zustand` | `5` | 15 个独立 `Store` 驱动状态 |
| 前端 | `@xterm/xterm`（xterm.js）| `6` | `Terminal` 泳道浏览器端渲染 |
| **后端** | `Node.js` + `TypeScript` | `22.x` / `5.7` | 运行时与类型系统 |
| 后端 | `Express` | `5` | 15 路 `REST API`（`:3001`） |
| 后端 | `ws` | `8` | 2 路 `WebSocket`（`/ws/chat`、`/ws/terminal`） |
| 后端 | `@anthropic-ai/claude-agent-sdk` | `0.2.63` | `Chat` 泳道：`spawn` `Claude Code CLI` 子进程 |
| 后端 | `node-pty` | `1.1` | `Terminal` 泳道：`spawn` `bash`/`zsh` `PTY` |

</div>

&emsp;&emsp;这张表回答的是"项目用了什么"——是一份**零件清单**。下一节的全局鸟瞰图回答的是"这些零件怎么被组织成层和通道"——那才是**装配图**。两者视角互补，先看清零件再看装配方式。

&emsp;&emsp;表里最值得记住的是**两条进程链的技术分工**：`Chat` 泳道走 `@anthropic-ai/claude-agent-sdk` 拉起 `Claude Code CLI` 子进程，`Terminal` 泳道走 `node-pty` 拉起 `bash`/`zsh` `PTY` 子进程——这是 3.2 节双泳道架构的最底层依据。

### 3.1 全局鸟瞰图：6 层分层与 3 条通信通道

&emsp;&emsp;上一节我们把项目用到的核心技术摊在了桌上——你现在知道"用了什么"。这一节要回答的是："**这些零件怎么被组织成一个能运转的系统？**"——把第二章看到的"3 栏 + 6 `Modal` + 15 路 `REST` + 2 路 `WebSocket`"放进一张统一的架构图里，看清每个功能落在哪一层、数据怎么流动。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140033235.png" width=70%></div>

&emsp;&emsp;从上到下，整个项目分为 6 层：

&emsp;&emsp;**第 1 层 · 浏览器组件层**：`React 19` + `Vite` + `Tailwind CSS v4`，由 15 个独立 `Zustand` Store 驱动状态。左栏 `Sidebar`（Files / Search / Checkpoints）、中栏 `ChatPanel`（对话 + 工具卡片 + `HIL` 弹窗）、右栏 `RightPanel`（Monitor / 拓展 / Agent / 终端）三个区域各自独立渲染，互不阻塞。6 个全局 `Modal`（History / FileView / Settings / FolderBrowser / SkillBrowser / CreateSkill）浮在顶层。

&emsp;&emsp;**第 2 层 · 通信层**：浏览器和后端之间有 3 条通信通道（这里按**协议类型**分 3 类，不是按数量计：`/ws/chat` 对话流、`/ws/terminal` 终端 I/O、`REST API` 配置管理——前两类各走一条 `WebSocket`，第三类是 HTTP）——2 路 `WebSocket`（`/ws/chat` 走对话流、`/ws/terminal` 走终端 I/O）和 15 路 `REST API`（走配置管理：agents / files / mcp / memory / sessions / system / teams / workflows 等）。<font color=red>一个关键设计决策：实时流式数据走 `WebSocket`，非实时配置操作走 `REST`</font>——这种分离让前端可以独立处理"对话中断了但配置页还能用"的场景。

&emsp;&emsp;**第 3 层 · 路由层**：后端 `Express` 服务的入口。`WebSocket` 路由在 `server/src/websocket/` 下（`chatHandler.ts` + `terminalHandler.ts`），`REST` 路由在 `server/src/routes/` 下（8 个路由文件，每个文件对应一个功能域，合计对外暴露约 15 个端点——即前文反复出现的"15 路 `REST`"）。<font color=red>路由层只做参数校验和调用转发，不含业务逻辑</font>——这是 `Fufan-CC` 后端分层的核心原则。

&emsp;&emsp;**第 4 层 · 服务层**：所有业务逻辑都在 `server/src/services/` 下。最核心的两个：`ClaudeAgentService`（封装 SDK 调用 + `HIL` 权限 + checkpoint）和 `sessionManager`（读取 `JSONL` 事实源 + 4 件加工）。其他服务如 `mcpService`、`memoryService`、`teamService` 等分别对应拓展层的各个功能域。

&emsp;&emsp;**第 5 层 · 进程层**：通过 `claude-agent-sdk` 的 `query()` spawn `Claude Code CLI` 子进程（Chat 泳道），通过 `node-pty` spawn `bash`/`zsh` shell 子进程（Terminal 泳道）。这两个进程互不感知。

&emsp;&emsp;**第 6 层 · 本地文件系统**：`CLI` 进程把对话 transcript 落盘到 `~/.claude/projects/<hash>/*.jsonl`，shell 进程直接操作用户工作目录。`Fufan-CC` 自己不存数据，只读 `CLI` 落盘的文件——这就是第五章要讲的"零侵入哲学"的根基。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Fufan-CC Flow 后端 REST API 路由一览</font></p>
<div class="center">

| 路由文件 | 功能域 | 典型端点 |
|----------|--------|----------|
| `agents.ts` | Agent 管理 | `GET /agents`、`POST /agents` |
| `files.ts` | 文件操作 | `GET /files/tree`、`GET /files/content` |
| `mcp.ts` | MCP Server 管理 | `GET /mcp/servers`、`POST /mcp/servers` |
| `memory.ts` | Memory 读写 | `GET /memory`、`PUT /memory` |
| `sessions.ts` | 会话管理 | `GET /sessions`、`GET /sessions/:id/messages` |
| `system.ts` | 系统状态 | `GET /system/health`、`GET /system/env` |
| `teams.ts` | Agent Teams | `GET /teams`、`POST /teams` |
| `workflows.ts` | 工作流编排 | `GET /workflows`、`POST /workflows` |

</div>

&emsp;&emsp;这张 REST 路由表对应的就是第二章 2.5 节拓展系统和 2.6 节 Agent 系统背后的数据通道。每个路由文件只做参数校验，真正的业务逻辑下沉到同名的 `service` 文件——比如 `routes/mcp.ts` → `services/mcpService.ts`。

> **【源码锚点】**：`server/src/routes/`（8 个路由文件）、`server/src/services/`（核心业务逻辑）、`server/src/websocket/`（2 个 WS handler）

### 3.2 双泳道 5 层架构

&emsp;&emsp;现在我们把刚才看到的"形状"还原成"内部结构"。`Fufan-CC` 的后端不是一条流水线，而是**两条并行的进程链**——我们叫它"双泳道"。每条泳道都由 5 层组成（浏览器 → WebSocket → Node 服务层 → CLI 进程 → 本地文件系统），但中间的 `spawn` 方式和数据格式完全不同。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140033137.png" width=60%></div>

&emsp;&emsp;让我们逐层看清楚。从浏览器最顶层往下数：

&emsp;&emsp;**第 1 层 · 浏览器**：`Fufan-CC` 的前端是 `React 19` + `Vite` + `Tailwind CSS v4`，左侧聊天面板（文件 `ChatPanel.tsx`）和右侧终端面板（文件 `Terminal.tsx`，组件 `TerminalPanel`）是两个独立的 React 组件，状态走 `Zustand` Store——对话相关在 `chatStore`，终端开关等 UI 全局状态在 `uiStore`。这一层你不需要管细节，只要知道"前端有两个面板对应后端两条泳道"。

&emsp;&emsp;**第 2 层 · WebSocket**：前后端不是用 REST API 通信，而是各开一个长连接——`/ws/chat` 走聊天，`/ws/terminal` 走终端 I/O。为什么用 WebSocket？因为 CLI 输出是流式的（Claude 一个 token 一个 token 地吐文字 + 实时穿插工具调用 + 终端可能瞬间 `console.log` 一万行），HTTP 长轮询撑不住这种密度。

&emsp;&emsp;**第 3 层 · Node 服务层**：这是整个 `Fufan-CC` 后端的核心。Chat 泳道由 `chatHandler.ts`（WebSocket 路由）+ `ClaudeAgentService`（封装 SDK 调用 + EventEmitter）组成；Terminal 泳道由 `ptyService.ts`（封装 `node-pty` 调用）组成。这一层的职责是"把 WebSocket 消息翻译成 SDK / PTY 调用，再把 SDK / PTY 的输出翻译回 WebSocket 事件"。

&emsp;&emsp;**第 4 层 · 进程层**：这一层是真正的"干活进程"。Chat 泳道通过 `claude-agent-sdk` 的 `query()` 函数 `spawn` 出 `Claude Code CLI` 子进程（CLI 自带模型 API 调用、工具执行、文件读写、MCP 集成等完整能力）；Terminal 泳道通过 `node-pty` 包装 `spawn` 出一个 `bash` / `zsh` shell 子进程（用户在终端里敲什么命令都直接送给这个 shell 执行）。**这两个进程互相不知道对方存在**，应用层也不强行同步它们的状态。

&emsp;&emsp;**第 5 层 · 本地文件系统**：Chat 泳道的 CLI 进程会把每一次对话的完整 transcript 写到 `~/.claude/projects/<projectHashPath>/<sessionId>.jsonl`（一行一条消息的 JSON Lines 格式）；Terminal 泳道的 shell 进程则直接操作用户工作目录（用户在终端里 `vim foo.py` 就是真的改 `foo.py` 文件）。**第 5 层是真正的"事实源"**——后面第五章讲的"零侵入读 JSONL"就是直接读这一层。

&emsp;&emsp;**三个核心抽象**值得在这里点名，因为它们在后两个模块会反复出现：

&emsp;&emsp;**`ClaudeAgentService`**——Node 服务层的核心 `EventEmitter` 类，封装 `query()` 调用 + HIL 权限管理 + checkpoint / rewind 逻辑，源码在 `claudeAgentService.ts:1-607`。

&emsp;&emsp;**`sessionManager`**——本地文件系统层的"读取适配器"，专门负责读 `~/.claude/projects/` 下的 JSONL + `sessions-index.json`，做 4 件加工（路径哈希 / 索引解析 / 内部消息过滤 / file-history-snapshot 解析），源码在 `sessionManager.ts:1-970`。

&emsp;&emsp;**`chatHandler`**——WebSocket 层的 message dispatcher，把客户端发来的 `send_message` / `abort` / `permission_response` 等指令翻译成对 `ClaudeAgentService` 的方法调用，反过来把 `ClaudeAgentService` 的 EventEmitter 事件转成 WebSocket 推送，源码在 `chatHandler.ts:1-211`。

### 3.3 单条消息的飞行路径：8 步贯穿 5 层

&emsp;&emsp;前面那张架构图是静态的。但实际上 `Fufan-CC` 处理一条消息的过程是有时序的——一次用户输入要在 5 层之间走完整 8 步。我们把这条路径叫"飞行路径"。你需要把这 8 步背下来，因为后两章讲的所有技术点，都会回引到"飞行路径第 X 步"。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140033112.png" width=70%></div>

&emsp;&emsp;（8 步飞行路径示意——一次用户输入在 5 层间的完整时序，后两章所有讲解都会回引到这张图的某一步）

&emsp;&emsp;以一次最普通的对话为例（用户在浏览器里输入"读一下 `package.json` 然后告诉我用了哪些 React 版本"，按发送）：

&emsp;&emsp;**第 1 步**：浏览器的 `ChatPanel` 组件捕获用户输入，调 `chatStore.sendMessage()`，store 把消息序列化成 `{"action": "send_message", "payload": {"prompt": "...", "runMode": "default"}}` 通过 WebSocket 发出去。

&emsp;&emsp;**第 2 步**：WebSocket 在后端 `chatHandler.ts:83` 的 `ws.on("message")` 回调里被接收，`switch (msg.action)` 走到 `case "send_message"` 分支。这里 `chatHandler.ts:106` 还会检查 `runMode === "plan"`——如果是 Plan Mode 会把 prompt 改写成 `"【PLAN MODE】请先分析需求并制定详细的执行计划，..." + 原 prompt`。

&emsp;&emsp;**第 3 步**：`chatHandler` 调 `claudeAgentService.start({prompt, projectPath, ...})`，这是从 WebSocket 层进入到 Node 服务层。

&emsp;&emsp;**第 4 步**：`ClaudeAgentService.start()` 在 `claudeAgentService.ts:97` 调用 `query({prompt, options})`，这是从 Node 服务层进入到进程层。`query()` 内部会 `spawn` 一个 Claude Code CLI 子进程，并返回一个 `AsyncIterable` 流。

&emsp;&emsp;**第 5 步**：CLI 子进程开始处理任务——它先把 prompt 发给 Claude 模型 API，模型回复说"我需要调 `Read` 工具读 `package.json`"，CLI 收到这个工具调用决策后，会先通过 `can_use_tool` 回调（对应 TS 侧的 `canUseTool`，详见 2.3 节的端对应说明）向上层请求权限（这就是 HIL 的入口，详见第 4.4 节）。

&emsp;&emsp;**第 6 步**：CLI 拿到批准后真的去读了 `package.json`（在第 5 层的本地文件系统），同时把读到的内容追加到 `~/.claude/projects/<hash>/<sessionId>.jsonl` 这个事实源文件里。

&emsp;&emsp;**第 7 步**：CLI 把"工具调用结果 + 模型继续生成的回答"通过 stdio 流式回吐给 `query()` 返回的 `AsyncIterable`，`ClaudeAgentService.consumeStream()` 在 `claudeAgentService.ts:281-290` 用 `for await (const msg of stream)` 消费它，然后通过 EventEmitter `emit` 出各种事件（`assistant_text` / `tool_use_start` / `tool_use_result` / `context_usage` 等）。

&emsp;&emsp;**第 8 步**：`chatHandler` 在 `chatHandler.ts:35-42` 监听这些 EventEmitter 事件，每收到一个就调 `forward(event, data)` 通过 WebSocket 推给浏览器；浏览器的 `ChatPanel` 通过 `useWebSocket` Hook 收到事件，更新 `chatStore`，React 重新渲染——用户在 UI 上就看到"Claude 正在打字 → 弹出工具调用卡片 → 工具调用完成 → Claude 继续打字 → 完整回复"的体验。

&emsp;&emsp;现在你应该能看清，**双泳道架构的 5 层 + 飞行路径的 8 步**就是整个 `Fufan-CC` 的骨架。后两个模块只是在这个骨架的不同位置做"放大镜"：第四章放大第 3-7 步（UI ↔ SDK 映射 + Python 复现 + HIL 桥接），第五章放大第 6 步那个 JSONL 事实源（零侵入读取 + 4 件加工）。

&emsp;&emsp;**第三章到这里完成**——20 分钟里你拿到三件实在的产出：一张把项目 6 层分层和 3 条通信通道摊平的全局鸟瞰图，一张能讲清 `Fufan-CC` 双泳道 5 层结构的架构图，一条能跟着讲清"一次发送怎么从浏览器走到 JSONL"的 8 步飞行路径。这是本课的**地图**，第四章和第五章接下来所有细节都会回指这张图里的某一层、某一步。如果你现在脑子里能凭空画出"`Chat/Agent 泳道` 走 `Agent SDK` / `Terminal 泳道` 走 `node-pty` / 第 6 步 CLI 把数据落到 `~/.claude/projects/`"——那第三章的目标就达到了。

---

## <center>第四章：UI ↔ SDK 映射 + Python 复现</center>

&emsp;&emsp;前一个模块我们看清了"路径"——一条消息怎么在 5 层 8 步之间走完。现在我们要做的是**对着这条路径的中间几步做"放大镜"**：浏览器 UI 上的每一个交互元素（输入框、模式切换、permission 弹窗、token 进度条……），对应到后端 `claude-agent-sdk` 的哪个参数、哪个事件、哪行代码？

&emsp;&emsp;这是整门课信息密度最高的模块。我们会按六个递进的小节展开：先用一张 10 行的 UI ↔ SDK 映射表把所有交互一次性铺开并量化标注 4 类后端转译（4.1）；再用 Python 真跑 `query()` 流看清 5 种 SDK 消息的流转顺序（4.2）；接着把 SDK 原始消息归一化成稳定的后端事件协议（4.3）；然后聚焦最难的 HIL 异步桥接（4.4）；再用代码演示 Session 续聊、分叉和 Checkpoint 文件回滚（4.5）；最后把所有模块串成一个 WebSocket 驱动的最小 Web 产品（4.6）。

&emsp;&emsp;进入正题之前**先花两分钟把 Python 环境装齐**（下一节 4.0），后面 4.2 起的所有代码块都会真跑——`pip install` 没做完跑代码会直接报包找不到。

### 4.0 开始之前：Python 环境与依赖

&emsp;&emsp;从这一节开始我们要**真跑 Python 代码**——用 `claude-agent-sdk` 的 Python 实现去复现项目 TypeScript 端的 SDK 调用机制。在执行下面任何代码块之前，请先用一条命令把课件依赖装齐。

&emsp;&emsp;**Python 版本要求**：`3.10` ～ `3.12`（课件 `harness` 环境实测 `3.11.15`）。如果你本机有多个 Python 版本，**强烈建议**用 `conda` 或 `venv` 创建一个独立环境再装依赖，避免污染系统 Python。

&emsp;&emsp;**容易踩的坑**：PyPI 上的 `claude-agent-sdk` 是 Anthropic 官方 Python SDK，与第三章技术栈表里的 `@anthropic-ai/claude-agent-sdk`（npm 包，项目 TypeScript 端用）**是同一 SDK 的双语言实现，版本号独立演进**——所以 PyPI 端是 `0.2.87` 而 npm 端是 `0.2.63` 不算异常。

In [ ]:
# 一次性安装课件依赖（包含 Jupyter 环境 + claude-agent-sdk + websockets）
# 国内网络如果直连 PyPI 慢，把下一行替换为带镜像的版本：
#   !pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install -r requirements.txt

&emsp;&emsp;装完之后我们**核对一下**：Python 版本是否在 `3.10` ～ `3.12` 区间、`claude-agent-sdk` 与 `websockets` 是否就位、`Claude Code CLI` 是否在 `PATH` 里能找到。下面的代码块只 `print` 出实际版本，**不做强 `assert`**——课件已知有些 Python `patch` 版本下游兼容良好，强等号会误伤。

In [3]:
import sys
import shutil
from importlib.metadata import version, PackageNotFoundError

# 打印 Python 主版本，下面 WARN 会判断它是否落在 3.10-3.12 区间
print(f'Python    : {sys.version.split()[0]}')

# 逐个核对第四章真用到的两个 PyPI 包是否就位（jupyter 三件套是开发环境基线，本地有 Jupyter 就一定有）
for pkg in ('claude-agent-sdk', 'websockets'):
    try:
        print(f'{pkg:<17s}: {version(pkg)}')
    except PackageNotFoundError:
        print(f'{pkg:<17s}: NOT INSTALLED — 请先跑上面的 pip install')

# 检测 Claude Code CLI 是否在 PATH —— 没有它，claude-agent-sdk 起不来 subprocess
cli_path = shutil.which('claude')
if cli_path:
    print(f'claude CLI : {cli_path}')
else:
    print('claude CLI : 未找到 — claude-agent-sdk 会找不到 CLI 进程，先回到第一章把 Claude Code CLI 装上')

# 提示而非阻断：harness 实测 Python 3.11.15 / sdk 0.2.87 / websockets 16.0
py_major_minor = sys.version_info[:2]
if py_major_minor < (3, 10) or py_major_minor > (3, 12):
    print(f'\n[WARN] 当前 Python {py_major_minor[0]}.{py_major_minor[1]} 不在课件验证过的 3.10-3.12 范围内，未实测兼容性')

Python    : 3.11.15
claude-agent-sdk : 0.2.87
websockets       : 16.0
claude CLI : /Users/mac/.local/bin/claude


&emsp;&emsp;**结果判读**：上面这段脚本只是一次**轻量自检**，不阻断后续执行——拿到输出后请自己对照一下：

&emsp;&emsp;**1）`Python` 落在 `3.10` ～ `3.12` 区间**——如果脚本尾部出现 `[WARN]` 行，说明你的 Python 主版本不在课件验证过的范围里；课程后面的代码大概率还能跑，但遇到包导入异常时优先怀疑这条。

&emsp;&emsp;**2）`claude-agent-sdk` 和 `websockets` 都打印出版本号**——任何一行显示 `NOT INSTALLED`，回到上一格重跑 `!pip install -r requirements.txt`；如果反复失败，多半是 PyPI 直连慢，换镜像版本（注释里给了清华源 URL）。

&emsp;&emsp;**3）`claude CLI` 给出一个绝对路径**——如果显示"未找到"，说明 `Claude Code CLI` 没装或不在 `PATH`，回到第一章的 CLI 安装步骤补齐。SDK 本质是 `subprocess.spawn` 拉起这个 CLI 进程，没它一切免谈。

### 4.1 UI ↔ SDK 映射表：10 行 + 4 类映射类型

&emsp;&emsp;我们先把表铺开，然后再讲为什么"映射类型"列是这门课最重要的认知工具。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Fufan-CC UI 交互 ↔ Claude Agent SDK 参数映射表</font></p>
<div class="center">

| # | UI 位置 | UI 表现 | SDK 参数 / 事件 | Python 等价 | 映射类型 |
|---|---------|---------|----------------|------------|---------|
| 1 | 聊天输入框 | 用户输入文字按发送 | `query(prompt=..., options=...)` | `async for m in query(prompt="...", options=opts)` | 纯 SDK |
| 2 | 模式下拉 → "Plan" | 切换到 Plan Mode | **prompt 前缀改写**（非 SDK 参数）| `query(prompt="【PLAN MODE】..." + 原 prompt, ...)` | **prompt 改写** |
| 3 | 模型下拉 | 选 Sonnet / Opus | `options.model=MODEL`（MODEL 由 requirements.txt 锁定，见环境准备 cell） | `ClaudeAgentOptions(model="...")` | 纯 SDK |
| 4 | Effort 下拉 | low / medium / high / max | `options.effort="high"` | `ClaudeAgentOptions(effort="high")` | 纯 SDK |
| 5 | Permission 弹窗 | 工具调用时弹出 Allow / Deny | `options.can_use_tool=async_handler` + `permission_mode="bypassPermissions"` | `ClaudeAgentOptions(can_use_tool=h, permission_mode="bypassPermissions")` | 纯 SDK + 后端调度 |
| 6 | Auto-approve（自动批准）| 内置工具自动放行 | **后端 AUTO_APPROVE_TOOLS 白名单**（非 SDK 参数）| Python 自己实现 `if tool in WHITELIST: return PermissionResultAllow()` | **后端调度** |
| 7 | Token 进度条 | 实时上下文使用率 | `context_usage` 事件（**Fufan-CC 自造 WS event**）| Python 在 stream 中提取 `message.usage` 自己 emit | **后端加工** |
| 8 | "/compact" 命令 | 上下文压缩 | `context_compact` 事件（CLI 主动 emit）| 流里收到 `type=context_compact` 时处理 | 纯 SDK + 后端加工 |
| 9 | Session 列表 | 历史会话切换 | `options.resume=sessionId` / `options.fork_session=True` | `ClaudeAgentOptions(resume="...", fork_session=True)` | 纯 SDK |
| 10 | Checkpoint 时间线 | 文件版本回滚 | `options.enable_file_checkpointing=True` + `client.rewind_files(uuid)`（Python 端实例方法）+ `sessionManager.getSessionCheckpoints()` fallback | 三合一：SDK 标志 + 流方法 + 自实现 fallback | UI 聚合 |

</div>

&emsp;&emsp;现在到了**杀手锏认知**的时间——表里那个"映射类型"列。当我们把 10 行扫一遍，会发现一个反直觉的事实：**只有 4 行是"纯 SDK"——也就是你能在官方文档里查到的标准参数。剩下 6 行都涉及后端转译**。这 6 行可以归纳为 4 类后端转译：

&emsp;&emsp;**第一类 · prompt 改写（第 2 行）**：UI 上看起来是个"参数切换"（选 Plan Mode），但 SDK 根本没有 `planMode` 这个参数。后端干的事是**把 prompt 字符串前面拼一段"【PLAN MODE】请先分析需求..."**。我们看源码就一清二楚：

```typescript
// chatHandler.ts:105-110
let prompt = p.prompt as string;
const runMode = (p.runMode as string) || "default";
if (runMode === "plan") {
  prompt =
    "【PLAN MODE】请先分析需求并制定详细的执行计划，" +
    "不要修改任何文件，不要执行任何命令，只输出计划。  " +
    prompt;
}
```

&emsp;&emsp;**这段代码的隐含意义很重要**：Plan Mode 是后端"骗"模型的——通过往 prompt 前面塞硬性指令让 Claude 自己选择"只规划不执行"。它**不是**通过 SDK 的 `permission_mode` 之类的硬性约束机制实现的（如果你试图在 Python 里找 `planMode` 这个 SDK 参数，会查不到任何文档，因为它根本不存在）。

&emsp;&emsp;**第二类 · 后端调度（第 6 行）**：UI 上看到"内置工具自动批准"这个体验，看起来很像 SDK 提供的功能。实际上 SDK 只给你 `can_use_tool` 这一个回调，至于"哪些工具自动批准、哪些工具弹窗"，完全是后端自己定的白名单：

```typescript
// chatHandler.ts:11-15
const AUTO_APPROVE_TOOLS = new Set([
  "Read", "Glob", "Grep", "WebSearch", "WebFetch",
  "TodoRead", "Task", "Agent", "TodoWrite",
  "NotebookRead", "LS",
]);
```

&emsp;&emsp;后端在 `permission_request` 事件触发时，先查这个白名单——命中就直接 `resolvePermission(requestId, "allow")` 不弹窗，没命中才转发给前端等用户决策。这个白名单是 `Fufan-CC` 自己挑的"只读、低风险"工具集，**换一个项目可能就是完全不同的白名单**（比如更激进的项目可能把 `Edit` 也加进去）。

&emsp;&emsp;**第三类 · 后端加工（第 7 行）**：这是最容易踩坑的一类。UI 上看到"实时 token 进度条"，很多人会以为 `claude-agent-sdk` 自带一个 `onContextUsage` 回调——**它没有**。`context_usage` 这个名字是 `Fufan-CC` 自己造的 WebSocket 事件。后端干的事是**在消费 SDK 流的过程中，从 `message.usage` 字段里挖出 token 数，然后自己 `emit` 出 `context_usage` 事件**。代码里有 3 个 emit 点：

```typescript
// claudeAgentService.ts:408+548+556 三 emit 点之一 —— 408 行（assistant message 收到时）
const usage = message.usage as Record<string, unknown> | undefined;
if (usage) {
  logger.debug(`[${sessionId}] context_usage: input=${usage.input_tokens} ...`);
  this.emit("context_usage", { sessionId, usage });
}

// claudeAgentService.ts:408+548+556 三 emit 点之二 —— 548 行（stream_event message_start 时）
if (eventType === "message_start") {
  const message = event.message as Record<string, unknown> | undefined;
  const usage = message?.usage as Record<string, unknown> | undefined;
  if (usage) this.emit("context_usage", { sessionId, usage });
}

// claudeAgentService.ts:408+548+556 三 emit 点之三 —— 556 行（stream_event message_delta 时）
if (eventType === "message_delta") {
  const usage = event.usage as Record<string, unknown> | undefined;
  if (usage) this.emit("context_usage", { sessionId, usage });
}
```

&emsp;&emsp;三个 emit 点对应三个不同的更新时机——消息完成、流开始、流中途增量——这样前端的 token 进度条才能"实时"刷新。**如果你在 Python 里复现这个 UI 体验，你也得自己写 3 个 emit 点**，SDK 不会替你做。

&emsp;&emsp;**第四类 · UI 聚合（第 10 行）**：Checkpoint 时间线这个 UI，背后是 3 个能力的拼装——SDK 的 `enable_file_checkpointing=True` 启用追踪、`client.rewind_files(uuid)` 做实际回滚、`sessionManager.getSessionCheckpoints()` 做 fallback（兜底路径——当 stream 已经结束时，从 JSONL 里重建 checkpoint 列表）：

```typescript
// claudeAgentService.ts:106
enableFileCheckpointing: true,

// claudeAgentService.ts:180-193（rewindFiles 包装）
async rewindFiles(sessionId: string, messageUuid: string, dryRun = false) {
  const stream = this.activeStreams.get(sessionId);
  if (!stream) throw new Error(`No active stream for session ${sessionId}. Use fallback rollback.`);
  const result = await stream.rewindFiles(messageUuid, { dryRun });
  return result;
}

// sessionManager.ts:727（fallback 入口）
async getSessionCheckpoints(sessionId: string): Promise<CheckpointsResult> {
  // 当 stream 已结束、无法用 SDK 的 rewindFiles 时，
  // 直接读 JSONL 里的 file-history-snapshot 重建 checkpoint 列表
}
```

&emsp;&emsp;**UI 聚合的本质是"用户看到的一个功能 = 后端拼装的 N 个能力"**。前端只看到一条"checkpoint 时间线"，后端要决定"现在该用 SDK 主路径还是该用 fallback 兜底路径"。

> **【杀手锏认知】**：4 类后端转译（**prompt 改写 / 后端调度 / 后端加工 / UI 聚合**）告诉你一个核心事实——**UI 上看到的东西，不一定是 SDK 直接给的**。下次你看任何"CLI 集成进 Web"项目，请用这 4 类去做归纳：是哪一类转译让 UI 这个交互成立？这个习惯能让你**少走很多弯路**。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140055130.png" width=60%></div>

&emsp;&emsp;（4 类后端转译概念模型——横轴区分"SDK 原生能力 vs 后端自造逻辑"，纵轴区分"改 prompt/数据 vs 改调度/呈现"，10 行映射表里的每一行都可以落进其中一个象限）

&emsp;&emsp;有了"4 类后端转译"这把分类工具，我们再补充两个常被忽略但同样重要的视角——一个是**反过来看：UI 上的"按钮"在后端最终去了哪里**，另一个是**深一步看：哪些"按钮"看起来已生效、实际后端根本没接**。这两个视角合起来就是判断"产品层 vs SDK 层"的实战技能。

&emsp;&emsp;**视角一 · 按钮的 3 个去向**：如果你把上面 10 行映射表反过来读——从 UI 输入侧看，每个按钮的值最终只会去 3 个地方：① 直接进 `ClaudeAgentOptions` 字段（如 `model` / `effort` / `resume` 都是 SDK 已定义的字段，第 1/3/4/9 行）；② 被后端 prompt 包装（如 Plan Mode，第 2 行——后端在调 SDK 之前对 prompt 字符串做改写）；③ **作为环境变量或服务端配置注入**（如 `ANTHROPIC_API_KEY` / `HTTP_PROXY` 在后端 `process.env` 里读、`CLAUDE_CODE_EFFORT_LEVEL` 在 CLI 进程启动时通过环境变量传入）——这一类**不在映射表里**因为它根本不暴露给 UI，但它跟前两类一样会真实影响 CLI 行为。后端代码里能看到 `delete process.env.CLAUDECODE` 这种"先清掉嵌套会话标记再启动 CLI"的细节，就是第 3 类去向的典型表现。

&emsp;&emsp;**视角二 · 按钮真假生效判断**：UI 上有控件 ≠ 后端真接了。看 `InputBar.tsx:15-16` 前端定义了 3 个模式：`default` / `plan` / `edit`，`InputBar.tsx:116-120` 也把 `runMode` 真的塞进了发送 payload。但翻到后端 `chatHandler.ts:105-110`——你只会看到 `if (runMode === "plan")` 这一个分支，**`edit` 分支完全不存在**。也就是说，UI 上 Edit 模式只是改了个图标和 store 状态，请求到了后端之后就被"静默丢弃"，CLI 跑出来跟 default 模式行为完全一样。<font color=red>这不是 bug，是 Fufan-CC 当前的真实状态——前端先把 UI 占位放上去，等待后端接入</font>。学员的训练点不在"修这个 bug"，而在养成"按钮真假生效判断"的习惯：**任何时候看到一个 Web 控件，先 grep 一下后端有没有处理这个字段——前端 emit 字段 ≠ 后端有 handler**。这个判断习惯比"读懂 4 类后端转译"更基础，但更容易被忽略。

> **【踩坑预警】**：怎么判断自己是不是踩了"静默丢弃"这坑？两个验证手段——① 在 `chatHandler.ts:105-110` 加一行 `console.log("[chatHandler] runMode =", runMode)`，发消息后看终端打印的 `runMode` 是不是真的传到了对应分支；② 在前端 Network 面板查 WebSocket 帧，确认 payload 里 `runMode: "edit"` 真的发出去了，再去后端 grep `case "edit"` 看 handler 是否存在。**修复方向**：要让 Edit 模式真正生效，需要在 `chatHandler.ts` 的 `if (runMode === "plan")` 后增加 `else if (runMode === "edit")` 分支，按 Plan 模式同样的 prompt 改写思路给 Edit 加专属前缀。

### 4.2 query 流式引擎与消息分发

&emsp;&emsp;上一节我们用映射表把 10 个 UI 交互一次性铺开，"纯 SDK"那一列里反复出现的就是 `query()`。现在我们从"看表格"切换到"真跑代码"——用 Python 实际跑通 SDK 的 `query()` 流，亲眼看清 5 种消息类型按顺序流出来。`query()` 是整个项目的心脏：**所有功能——对话、工具调用、session resume、权限回调——最终都通过这一个异步生成器入口**，没有例外。

&emsp;&emsp;我们从最简单的调用开始。下面这个 cell 只请求模型回一句话，目的是拿到 `SystemMessage(init)` 和 `ResultMessage` 这两端：

In [4]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions,
    SystemMessage, AssistantMessage, UserMessage, ResultMessage,
    TextBlock, ToolUseBlock, ToolResultBlock,
)

opts = ClaudeAgentOptions(max_turns=1)

async def run_dispatch():
    """跑一次 query，按消息类型分发处理"""
    # async for 是必须的，不能 list() 强转（query 返回异步生成器，强转会阻塞事件循环）
    async for msg in query(prompt="用 5 个中文字回答：什么是 LLM？", options=opts):
        # subtype='init' 是拿 session_id 的唯一时机——后续 resume/fork_session 都靠它
        if isinstance(msg, SystemMessage) and msg.subtype == "init":
            sid = msg.data.get("session_id", "?")
            print(f"[init]   sid={sid[:8]}...  model={msg.data.get('model')}")
        elif isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, TextBlock):
                    print(f"[text]   {block.text[:80]}")
                elif isinstance(block, ToolUseBlock):
                    print(f"[tool]   {block.name}({list(block.input.keys())})")
        elif isinstance(msg, ResultMessage):
            cost = msg.total_cost_usd or 0
            print(f"[result] cost=${cost:.4f}  duration={msg.duration_ms}ms")

await run_dispatch()

[init]   sid=520465de...  model=claude-opus-4-7[1m]
[text]   大语言模型
[result] cost=$0.3099  duration=8075ms


&emsp;&emsp;跑完这个 cell，你会注意到三件事——把这三件事记牢，后面写任何 SDK 集成都用得上。**第一**，`query()` 返回的是异步生成器，必须用 `async for` 迭代，不能直接 `list()` 强转（会挂住）。**第二**，`SystemMessage(init)` 是整个流里**最先到达**的消息，也是拿 `session_id` 的唯一时机——后续如果要做 `resume` 或 `fork_session`，就靠这里记录下来的 `session_id`。**第三**，`ResultMessage` 是费用和 token 数的来源。注意它是纯 `@dataclass`，字段直接挂在对象上，不是 `.data` 字典——`cost` 取 `msg.total_cost_usd`（`float | None`），耗时取 `msg.duration_ms`（`int`），token 数取 `msg.usage`（结构为 `{"input_tokens": N, "output_tokens": N}`）。如果你要做计费日志或进度条，从这些直接属性取。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140058530.png" width=60%></div>

&emsp;&emsp;只跑一句对话，我们只看到了 5 种消息里的 3 种：`SystemMessage(init)` / `AssistantMessage(TextBlock)` / `ResultMessage`。另外 2 种——`AssistantMessage(ToolUseBlock)` 和 `UserMessage(ToolResultBlock)`——只有当模型真正调用工具时才出现。下面这个 cell 让模型读一个文件，强制触发工具调用，这样 5 种消息就能按顺序全部流出来：

In [5]:
opts = ClaudeAgentOptions(
    max_turns=5,
    # bypassPermissions 把权限决策完全交给 can_use_tool 回调，避免每次工具调用都弹确认
    permission_mode="bypassPermissions",
)

async def run_with_tool():
    """让模型读 metadata.json，观察 5 种消息按顺序流出"""
    # prompt 显式要求读文件，强制触发 ToolUseBlock + ToolResultBlock 两类块流出
    async for msg in query(prompt="读 metadata.json 文件，一句话告诉我这是什么文件", options=opts):
        if isinstance(msg, SystemMessage) and msg.subtype == "init":
            print(f"[1·init]     sid={msg.data.get('session_id', '?')[:8]}...")
        elif isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, ToolUseBlock):
                    print(f"[2·tool_use] {block.name}  input_keys={list(block.input.keys())}")
                elif isinstance(block, TextBlock):
                    print(f"[4·text]     {block.text[:80]}")
        elif isinstance(msg, UserMessage):
            for block in msg.content:
                if isinstance(block, ToolResultBlock):
                    print(f"[3·tool_res] content_len={len(str(block.content))}")
        elif isinstance(msg, ResultMessage):
            print(f"[5·result]   cost=${(msg.total_cost_usd or 0):.4f}")

await run_with_tool()

[1·init]     sid=6055292d...
[2·tool_use] Glob  input_keys=['pattern']
[3·tool_res] content_len=20
[2·tool_use] Read  input_keys=['file_path']
[3·tool_res] content_len=123
[2·tool_use] Glob  input_keys=['pattern', 'path']
[3·tool_res] content_len=494
[2·tool_use] Read  input_keys=['file_path']
[3·tool_res] content_len=7811
[4·text]     这是课件生成流水线（courseware-pipeline）的元数据配置文件，定义了"Fufan-CC Flow 2 小时速通课"的课程主题、学员画像、能力目标
[5·result]   cost=$0.7097


&emsp;&emsp;编号 `[1]` 到 `[5]` 就是这 5 种消息在真实流中的出现顺序。特别注意 `[3·tool_res]`：它是 `UserMessage` 而不是 `AssistantMessage`——SDK **自动把工具执行结果封装成一条 `UserMessage` 回灌给模型**，模型在下一轮才能看到工具的输出，从而生成 `[4·text]` 那条最终回复。这个"工具结果走 UserMessage 通道"的设计，正是你在后端里看到 `UserMessage` 时不要惊讶的原因。

&emsp;&emsp;把这 5 种消息的语义整理成表格，方便后面直接对照 Fufan-CC 后端代码查阅：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>SDK 消息类型与出现时序</font></p>
<div class="center">

| 序号 | 消息类型 | 触发时机 | 关键用途 |
|------|---------|---------|---------|
| 1 | `SystemMessage(init)` | 会话起手，流的第一条消息 | 拿 `session_id` 的唯一时机 |
| 2 | `AssistantMessage(ToolUseBlock)` | 模型决定调用某个工具时 | 对应 Fufan-CC UI 的 ToolCallCard |
| 3 | `UserMessage(ToolResultBlock)` | 工具执行完，SDK 把结果回灌给模型 | SDK 自动生成，后端按需二次加工 |
| 4 | `AssistantMessage(TextBlock)` | 模型基于工具结果生成最终回复 | 对应 UI 的聊天气泡 |
| 5 | `ResultMessage` | 会话收尾，最后一条消息 | 费用 / token 数 / 耗时的来源 |

</div>

&emsp;&emsp;这 5 种消息类型，正是 `claudeAgentService.ts:329-488` 里 `dispatch()` 函数要处理的全部输入。`dispatch()` 160 行代码，本质就是一条 `if-elseif` 链——我们刚才写的 `run_with_tool()` 就是它的 Python 最小版。下一节我们把这个"最小版"升级一步：加上归一化层，让你看清后端为什么不把 SDK 原始对象直接推给前端。

> **【踩坑预警】**：`permission_mode="bypassPermissions"` 会跳过所有工具权限确认，在开发调试时很方便。但在生产环境或课堂演示时，一定记得去掉这个参数——否则模型可以无障碍读写任意文件，包括敏感配置。

---

### 4.3 后端事件归一化：从 SDK 对象到产品协议

&emsp;&emsp;上一节我们亲眼看到了 5 种 SDK 消息按顺序流出来。但 Fufan-CC 后端**不会把这些 SDK 原始对象直接推给浏览器**——它在中间加了一层归一化，先把 SDK 对象转换成自己定义的稳定事件结构，再通过 WebSocket 推给前端。为什么要多这一层？答案很简单：**SDK 是外部依赖，字段名随版本升级可能改变；前端协议是内部契约，必须保持稳定**。如果不加这一层适配，SDK 每次改个字段名，前端就得跟着改——这是典型的"把外部不稳定性暴露给内部消费者"的设计问题。归一化层的作用，就是在这两个变化速率不一致的系统之间做缓冲。

&emsp;&emsp;先把映射关系铺出来，再看代码：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>SDK 消息 → 后端产品事件映射</font></p>
<div class="center">

| SDK 消息类型 | 归一化后的事件名 | 阶段含义 |
|-------------|----------------|---------|
| `SystemMessage(init)` | `session_init` | 会话建立，通知前端会话 ID 和模型信息 |
| `AssistantMessage(TextBlock)` | `assistant_text` | 文本输出，前端渲染聊天气泡 |
| `AssistantMessage(ToolUseBlock)` | `tool_use_start`（完整版另有 `tool_input_complete`，本课 Python 精简版略去） | 工具卡片展开，前端显示工具名和参数 |
| `UserMessage(ToolResultBlock)` | `tool_use_result` | 工具执行结果，前端更新工具卡片状态 |
| `ResultMessage` | `task_complete` | 会话收尾，前端更新费用和耗时展示 |

</div>

&emsp;&emsp;下面这个函数就是归一化层的 Python 精简版。它接收一条 SDK 消息，返回一个事件列表——一条 SDK 消息可能对应多个事件（完整版本 `dispatch()` 里 `ToolUseBlock` 还会额外触发 `tool_input_complete` 用于"工具参数流式输入完成"信号，本课精简版只演示 `tool_use_start`）：

In [6]:
def dispatch_message(msg):
    """把 SDK 原始消息转换成稳定的后端事件列表（Fufan-CC dispatch() 的 Python 精简版）"""
    events = []

    # SystemMessage(init) 对应 session 初始化事件，前端用 sessionId 做 resume 标识
    if isinstance(msg, SystemMessage) and msg.subtype == "init":
        events.append({
            "event": "session_init",
            "sessionId": msg.data.get("session_id"),
            "model": msg.data.get("model"),
        })

    elif isinstance(msg, AssistantMessage):
        for block in msg.content:
            if isinstance(block, TextBlock):
                # TextBlock → assistant_text，最常见的流式 token 输出
                events.append({"event": "assistant_text", "text": block.text})
            elif isinstance(block, ToolUseBlock):
                # ToolUseBlock → tool_use_start，前端用 toolName/toolInput 渲染工具卡片
                events.append({
                    "event": "tool_use_start",
                    "toolName": block.name,
                    "toolInput": block.input,
                    "toolUseId": block.id,
                })

    elif isinstance(msg, UserMessage):
        # 注意：工具执行结果被 SDK 封装成 UserMessage 回灌（而非 AssistantMessage）
        for block in msg.content:
            if isinstance(block, ToolResultBlock):
                events.append({
                    "event": "tool_use_result",
                    "toolUseId": block.tool_use_id,
                    "content": str(block.content)[:200],
                })

    elif isinstance(msg, ResultMessage):
        # ResultMessage 是流的最后一条，含本轮总成本/耗时，前端用来更新费用展示
        events.append({
            "event": "task_complete",
            "cost": msg.total_cost_usd,
            "duration": msg.duration_ms,
        })

    return events

print("dispatch_message() ready：真实项目会把这些 event 通过 WebSocket 推给浏览器")

# 真实调用示例（在 WebSocket 消息处理器中）：
# async for msg in query(prompt=user_text, options=opts):
#     events = dispatch_message(msg)
#     for event in events:
#         await websocket.send(json.dumps(event))

dispatch_message() ready：真实项目会把这些 event 通过 WebSocket 推给浏览器


&emsp;&emsp;看完这个函数，有两个设计决策值得专门说一下。**第一**，函数返回的是事件**列表**而不是单个事件——因为一条 `AssistantMessage` 里可能同时包含多个 `ToolUseBlock`（模型一次决定调多个工具），每个 Block 都要独立触发一条 `tool_use_start` 事件，前端才能分别渲染多个工具卡片。**第二**，`tool_use_result` 里截断了 `content` 长度（`[:200]`）——真实项目里工具返回的内容可能很长（比如 Read 了一个大文件），前端渲染卡片只需要摘要，完整内容留在模型上下文里就够了。这种"截断归一化"是后端加工层的常见做法，不做截断的话 WebSocket 消息体会过大。

&emsp;&emsp;**关键设计决策**：SDK 升级改了字段名，只要改 `dispatch_message()` 这一层适配函数，前端协议字段（`event` / `sessionId` / `toolName` / `toolUseId`……）完全不受影响。这个"稳定接口隔离外部变化"的思路，是真实项目里处理第三方 SDK 升级的标准手法。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140057086.png" width=60%></div>

&emsp;&emsp;在真实的 Fufan-CC 里，`dispatch_message()` 的每条输出事件会经过如下链路到达浏览器：`dispatch()` 函数 emit → `EventEmitter` 总线 → `chatHandler.ts` 里注册的监听器 → `WebSocket.send()` → 浏览器前端接收并渲染。我们写的 Python 版本只做到了"生成 event 字典"这一步；理解了这一步，后面看 TypeScript 源码时那条完整链路就只是"事件怎么传"的问题，而不是"事件是什么"的问题了。

> **【源码锚点】**：`claudeAgentService.ts:329-488`——`dispatch()` 函数完整实现，TypeScript 版本的归一化逻辑与本节 Python 精简版一一对应。重点关注 `AssistantMessage` 分支（`:370-430`）里对 `ToolUseBlock` 和 `TextBlock` 的分别处理，以及 `ResultMessage` 分支（`:460-488`）里 `cost_usd` 的提取方式。

### 4.4 HIL 异步桥接：CLI 暂停到 Web 弹窗的最小版

&emsp;&emsp;HIL 是整门课最难也最值得讲的一个机制。HIL 全称 **Human-in-the-Loop**——人类在工具调用链路中扮演决策角色。具体场景是：Claude 想调一个"高风险"工具（比如 `Edit` 改文件、`Bash` 跑命令），CLI 必须暂停下来等用户决定 allow 还是 deny。在终端里这件事很简单——直接 `print("Allow? [y/n]")` + `input()`；但在 Web 应用里就复杂了：CLI 在后端，用户在浏览器，中间隔着 WebSocket，**怎么让 CLI 真的"暂停"在那里等浏览器的回信**？

&emsp;&emsp;`claude-agent-sdk` 给了一个把权限决策完全外包给调用方的异步回调接口——但**选哪一个有讲究**。SDK 提供两条机制：(1) `can_use_tool` 回调字段，看名字像是给 HIL 用的，**但实际只在 CLI 的 permission rules 评估到 "ask" 分支才触发**——SDK 默认模式下 CLI 直接 `auto-allow` 所有工具，根本走不到 ask 分支，callback 永远不会被调；(2) `hooks` 字段的 `PreToolUse` 钩子，**无条件**拦截每一次工具调用，这才是 SDK 0.2.87 真正可用的 HIL 入口（参见 `claude_agent_sdk/types.py` 中 `can_use_tool` 字段 docstring 末段的官方说明）。本节用 `hooks` 的 `PreToolUse` 把"CLI 暂停"机制跑通——签名是 `async def hook(input_data, tool_use_id, context) -> dict`，CLI 每次想调工具都会先 `await` 这个 hook，**hook 返回的 dict 里 `permissionDecision` 字段决定 CLI 继续执行还是拒绝执行**。换句话说，"CLI 暂停"这件事仍是通过"`await` 一个未完成的 Future"实现的。

&emsp;&emsp;我们用一个 **HIL 最小版**把这个机制跑通。核心逻辑约 30 行（不含注释和空行；含注释和导入的整段约 55 行），**输入用 `input()` 模拟 Web 弹窗**（同步阻塞），实际 Fufan-CC 在生产里是用 `asyncio.Future` + WebSocket 异步等回信，但最小版用 `input()` 已经足够展示核心机制：

&emsp;&emsp;**先看骨架再看细节**——下面这段代码由 3 个部分组成：**导入与配置段** 引入 `ClaudeSDKClient` / `HookMatcher`、定义 `ClaudeAgentOptions` 里的 `hooks` 字段；**`pre_tool_hook` 函数体** 是 HIL 的心脏——把 CLI 的工具请求映射到一次 `input()` 阻塞，再把用户选择翻译回含 `permissionDecision: "allow"/"deny"` 的 hook output dict；**`run_hil` 主流程** 用 `async with ClaudeSDKClient(...)` 启动持久客户端、`await client.query(...)` 发送 prompt、`client.receive_response()` 异步迭代消费消息。读的时候先按 3 个函数/段落各扫一眼整体职责，再回到 hook 内部看怎么写——这样的"按职责切片"方式比记行号更适合你之后挪到自己代码库。

In [7]:
# HIL 最小版：用 PreToolUse hook 拦截每个工具调用，让用户决策
# 真跑前提：本机已安装 Claude Code CLI 并登录
#
# 注意：SDK 0.2.87 的 can_use_tool callback 只在 CLI 的 permission rules
# 评估为 "ask" 时才触发；CLI 在 SDK 模式下默认 auto-allow 所有工具，所以
# can_use_tool 实际很难拦到调用。要无条件拦每个工具，官方推荐用 hooks 的
# PreToolUse — 这才是 SDK 0.2.87 真正可用的 HIL 机制（参见 SDK types.py 中
# can_use_tool 字段 docstring 的最后一段）。

import asyncio
from claude_agent_sdk import (
    ClaudeSDKClient,         # streaming 模式 client（hooks 必须配它）
    ClaudeAgentOptions,      # 配置类（包含 hooks 字段）
    HookMatcher,             # hook 匹配器（matcher='*' 拦所有工具）
)

async def pre_tool_hook(input_data, tool_use_id, context):
    """
    HIL 权限 hook：CLI 每次想调工具都会 await 这个函数
    Args:
        input_data: dict，含 tool_name / tool_input 等
        tool_use_id: 本次工具调用的唯一 id
        context: HookContext（含 future 的 abort signal 支持）
    Returns:
        dict 形式的 hook output，permissionDecision 决定 allow/deny/ask
    """
    tool_name = input_data.get("tool_name", "?")
    tool_input = input_data.get("tool_input", {})
    print(f"\n[HIL] 工具请求: {tool_name}")
    print(f"[HIL] 参数: {tool_input}")
    # input() 同步阻塞，在最小版里就是"Web 弹窗"的模拟
    # 真实 Web 项目里这里要换成 asyncio.Future + WebSocket 等回信（参考下节差异表）
    decision = input("[HIL] 是否允许？(y/n): ").strip().lower()
    if decision == "y":
        # permissionDecision='allow' 让 CLI 真执行工具
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "allow",
            }
        }
    else:
        # permissionDecision='deny' + reason，CLI 会把 reason 喂给模型继续推理
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": f"用户拒绝调用 {tool_name}",
            }
        }

# 关键配置：hooks 字段注册 PreToolUse 拦截所有工具（matcher='*'）
# 不设 permission_mode（默认即可），hook 的 permissionDecision 优先级最高
opts = ClaudeAgentOptions(
    hooks={
        "PreToolUse": [HookMatcher(matcher="*", hooks=[pre_tool_hook])],
    },
    max_turns=2,
)

async def run_hil():
    """让 Claude 做一个会触发工具调用的任务，看 hook 被触发"""
    # 用 ClaudeSDKClient streaming 模式（hooks 跑在它的控制协议上）
    async with ClaudeSDKClient(options=opts) as client:
        await client.query("用 Write 工具在 /tmp/fufan-hil-demo1.txt 创建文件，内容写 hello")
        async for msg in client.receive_response():
            if hasattr(msg, "content"):
                for block in msg.content:
                    if getattr(block, "type", None) == "text":
                        print(block.text, end="", flush=True)
    print()

await run_hil()


[HIL] 工具请求: Write
[HIL] 参数: {'file_path': '/tmp/fufan-hil-demo1.txt', 'content': 'hello'}



&emsp;&emsp;**运行后你会看到一次完整的 HIL 流程**：Claude 决定调 `Write` 工具 → 你的终端会跳出 `[HIL] 工具请求: Write` 提示 + 完整的 `file_path` / `content` 参数 → 你按 `y` 回车 → CLI 继续执行 `Write` 真创建文件 → Claude 把"已创建"的总结输出。如果你按 `n`，Claude 会收到"用户拒绝调用 Write"这条 message，然后会换一种说法回答（比如告诉你"无法创建文件，请你手动创建"）。

&emsp;&emsp;这里有几个关键点必须讲清楚。第一，**hook 注册在 `hooks` 字段里，不在 `query()` 也不在 `can_use_tool`**——`opts = ClaudeAgentOptions(hooks={"PreToolUse": [HookMatcher(matcher="*", hooks=[pre_tool_hook])]})` 是唯一正确写法。`matcher="*"` 表示拦所有工具；如果只想拦危险工具，可以写 `matcher="Write|Edit|Bash"` 这种正则。第二，**不要把 `permission_mode="bypassPermissions"` 当作 hooks 的搭档**——它字面意思是"绕过 CLI 内部所有权限检查/弹窗"（SDK `types.py:1633`：`"Bypass all permission checks"`），是为"我完全信任模型让它跑就行"场景设计的；和 HIL "拦下每个工具让用户决策"是相反需求。本节代码保持默认 mode（不显式设 `permission_mode`），让 `hooks.PreToolUse` 走它自己的拦截路径（这与 4.2 节"调试时 bypassPermissions 很方便"不冲突——那是 `query()` 快速跑通的场景，HIL 是反向需求）。第三，**hook 返回值必须是 dict，且要带 `hookSpecificOutput.hookEventName="PreToolUse"`**——SDK 用这个字段做 hook 类型路由，缺它会被当成无效响应忽略；`permissionDecision` 可选 `"allow"` / `"deny"` / `"ask"` / `"defer"`，其中 `"deny"` 建议同时带上 `permissionDecisionReason` 字段告诉模型拒绝原因（不带也合法，但模型不知道为什么失败也学不到东西）。

&emsp;&emsp;现在我们用一段叙述把"`input()` 这种同步阻塞，在 Web 里到底变成什么"讲清楚——这是从 HIL 最小版通往 Fufan-CC 生产代码的关键认知。

&emsp;&emsp;**`input()` → `asyncio.Future` → WebSocket 挂起的三层类比**：在 HIL 最小版里，`input()` 是同步阻塞——Python 进程卡在那一行等用户键入。在真实 Web 项目里，这个"卡住"必须升级为异步——因为后端要同时服务多个用户的多个会话，不能一个会话卡住整个 Node 进程。解决方案是把"等用户决策"包装成 `asyncio.Future`（Python 端）或者 `Promise`（Node 端）。具体到 `Fufan-CC` 的实现：

&emsp;&emsp;后端 `claudeAgentService.ts:230-277` 的 `requestPermission()` 方法里，每次 CLI 触发权限请求（TS 端 SDK 仍走 `canUseTool` 字段；Python 复现侧因 SDK 0.2.87 行为差异改走 `hooks.PreToolUse`，两边在 CLI 端协议层等价），后端就 `new Promise((resolve) => {...})` 创建一个未完成的 Promise，把 `resolve` 函数存到 `this.pendingPermissions` Map 里（key 是 `requestId`），然后 `emit("permission_request", {...})` 通过 EventEmitter 发出去——`chatHandler` 监听到后再通过 WebSocket 推给前端弹窗。`chatHandler.ts:71-81` 的转发逻辑也值得单独看一眼，因为它体现了"后端调度"那一类映射的核心：

```typescript
// chatHandler.ts:71-81 — HIL permission_request 转发 + 白名单自动批准
claude.on("permission_request", (d: PermissionRequest) => {
  if (AUTO_APPROVE_TOOLS.has(d.toolName)) {
    // 安全工具自动批准
    claude.resolvePermission(d.requestId, "allow");
    logger.debug(`Auto-approved tool: ${d.toolName} (${d.requestId})`);
  } else {
    // 危险工具转发给前端等待用户确认
    forward("permission_request", d as unknown as Record<string, unknown>);
    logger.info(`Permission requested for ${d.toolName} (${d.requestId})`);
  }
});
```

&emsp;&emsp;**这就是 `Fufan-CC` 实现"内置工具自动批准 + 危险工具弹窗"双轨制的全部代码**——`AUTO_APPROVE_TOOLS` 白名单命中就直接 `resolvePermission(d.requestId, "allow")` 不通知前端，没命中就走 `forward()` 把请求推到 WebSocket。前端用户点了 Allow / Deny 后，前端发回 `{"action": "permission_response", "requestId": "...", "decision": "allow"}`，后端在 `resolvePermission(requestId, "allow")` 里从 `pendingPermissions` Map 取出之前存的 `resolve` 函数并调用它——**这一调，那个原本未完成的 Promise 就 resolve 了，`canUseTool`（TS 端 SDK 字段）回调返回，CLI 继续往下跑**。

&emsp;&emsp;**整条链路的核心是"挂起一个 Promise/Future，直到外部输入到达"**。Python 里完全可以用 `asyncio.Future` 做一样的事——你创建一个 `loop.create_future()`，把它的 `set_result` 存到字典里，hook 里 `await future`，外部消息到达时调 `future.set_result(decision)`，hook 就"醒过来"返回一个含 `permissionDecision: "allow"/"deny"` 的 hook output dict。这是 HIL 最小版升级到生产版的核心改造。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140057683.png" width=70%></div>

&emsp;&emsp;（HIL 异步桥接全链路——从 CLI 触发权限请求到 Promise 挂起、WebSocket 推送弹窗、用户决策回传、resolve Promise 让 CLI 继续，完整时序一图看清。图中标注了 `AUTO_APPROVE` 分支和 60 秒超时降级路径。**TS 端**：Fufan-CC 用 SDK 的 `canUseTool` 字段；**Python 复现侧**：本课用 `hooks.PreToolUse`，因为 SDK 0.2.87 默认 `auto-allow` 路径下 `can_use_tool` 不触发——两边在 CLI 端控制协议层等价）

&emsp;&emsp;为了让你清楚"最小版"和"生产版"到底差在哪儿，我们用一张 7 行差异表把所有关键点列出来：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>HIL 最小版 vs 生产版 7 处差异</font></p>
<div class="center">

| 差异点 | 最小版（HIL 最小版）| 生产版（Fufan-CC）|
|--------|----------------|--------------------|
| 等待机制 | `input()` 同步阻塞 | `asyncio.Future` / `Promise` 异步挂起，外部 resolve |
| 请求标识 | 无（一次只处理一个） | 必有 `requestId`（CLI 提供的 `toolUseID` 或自生成），用 Map 管理多并发请求 |
| 超时降级 | 用户不输入就永远卡住 | `setTimeout(60_000)` 超时自动 `behavior: "deny"`（`claudeAgentService.ts:257-264`）|
| 并发支持 | 单线程一次只能一个 | `pendingPermissions: Map<requestId, {resolve, suggestions}>` 支持多会话多请求并发 |
| 取消能力 | 无 | `AbortController` 集成，session 被 abort 时自动 deny 所有 pending 请求 |
| 自动批准 | 无 | `AUTO_APPROVE_TOOLS` 白名单命中直接 `resolve("allow")` 不通知前端（`chatHandler.ts:72-75`）|
| 通信通道 | `input()` 走标准输入 | `EventEmitter` → `WebSocket` 双向通信，前后端各自反序列化 JSON 消息 |

</div>

&emsp;&emsp;**这张表的意义不是让你照着把 HIL 最小版改成 100 行的"生产版"**——那是 Fufan-CC 的工作。它的意义是让你看清："最小版只要 35 行就能让 HIL 跑起来"，但"从最小版到生产可用"中间还隔着 7 个独立的工程维度（其中 6 个是最小版完全没有的新问题，第 7 个"通信通道"是从 stdin 升级到 WebSocket 的替换）。如果哪天你自己包一个 CLI 到 Web，这 7 个维度就是你的 checklist。

---

### 4.5 Session 生命周期：续聊、分叉与 Checkpoint

&emsp;&emsp;会话连续性是任何 Web 化 AI 工具的基础产品能力。没有它，用户每次打开对话框都是白板——没有上下文、没有历史、没有记忆。`claude-agent-sdk` 通过两个字段实现三种操作模式：`resume` 字段决定是"续聊"还是"分叉"，`enable_file_checkpointing` 开启文件快照机制，让 AI 写过的文件改动可以一键回滚。本节用 4 段代码完整演示这套机制。**注意**：`rewind_files` 还需额外配 `extra_args={"replay-user-messages": None}` 才能拿到正确的 `user_message_id`，否则 rewind 会报 "No file checkpoint found"——细节见下一段 Checkpoint 讲解。

&emsp;&emsp;我们先把三种模式的核心差异铺开：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>表 4.5-1 Session 三种操作模式</font></p>
<div class="center">

| 操作 | `ClaudeAgentOptions` 关键字段 | 结果行为 |
|:----:|:----:|:----:|
| 新建 | 默认（不传 `resume`） | 生成全新 `session_id` |
| 续聊 | `resume=<sid>` | 加载历史继续，`session_id` 不变 |
| 分叉 | `resume=<sid>` + `fork_session=True` | 以历史为起点派生新 `session_id`，原 `sid` 保留完整 |

</div>

&emsp;&emsp;关键认知在于：`session_id` 不是我们给 SDK 的，而是 SDK 通过 `SystemMessage`（`subtype="init"`）反向告知调用方的。下面的 helper 函数统一提取这个值，再用它演示三步状态转移。

In [8]:
# Session 生命周期三步演示：新建 → 续聊 → 分叉

from claude_agent_sdk import query, ClaudeAgentOptions, SystemMessage, AssistantMessage, TextBlock

async def run_and_extract(prompt, options):
    """跑一次 query，返回 (session_id, 最终文本)"""
    sid, text = None, ""
    async for msg in query(prompt=prompt, options=options):
        if isinstance(msg, SystemMessage) and msg.subtype == "init":
            sid = msg.data.get("session_id")
        elif isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, TextBlock):
                    text += block.text
    return sid, text.strip()

# Step 1：新建会话
opts = ClaudeAgentOptions(max_turns=1)
sid_a, text_a = await run_and_extract("请记住数字 42。一句话确认你记住了。", opts)
print(f"[新建] sid_a = {sid_a}")
print(f"       text  = {text_a}\n")

# Step 2：续聊（resume=sid_a，session_id 不变）
opts = ClaudeAgentOptions(max_turns=1, resume=sid_a)
sid_a2, text_a2 = await run_and_extract("我刚才让你记的数字是多少？", opts)
print(f"[续聊] sid_a2 = {sid_a2}")
print(f"       text   = {text_a2}")
print(f"       续聊验证：sid_a == sid_a2 → {sid_a == sid_a2}\n")

# Step 3：分叉（fork_session=True，派生新 sid）
opts = ClaudeAgentOptions(max_turns=1, resume=sid_a, fork_session=True)
sid_b, text_b = await run_and_extract("我刚才让你记的数字翻倍是多少？", opts)
print(f"[分叉] sid_b = {sid_b}")
print(f"       text  = {text_b}")
print(f"       分叉验证：sid_b == sid_a → {sid_b == sid_a}（应为 False）")

[新建] sid_a = ee3125bb-fe49-49bd-8620-c3c26dd49756
       text  = 已记住，数字是 42。

[续聊] sid_a2 = ee3125bb-fe49-49bd-8620-c3c26dd49756
       text   = 42。
       续聊验证：sid_a == sid_a2 → True

[分叉] sid_b = f894c1c8-973c-4023-92f8-cdb461080320
       text  = 84。
       分叉验证：sid_b == sid_a → False（应为 False）


&emsp;&emsp;三步执行完，有三个关键认知需要钉住：① `session_id` 是 SDK 在 `SystemMessage(init)` 里告诉你的，不是你指定的；② `resume` 让 CLI 加载该 session 的历史消息后接着跑，`session_id` 保持不变；③ `fork_session=True` 以历史为起点但派生一条全新的会话线，原 `sid_a` 完全不受影响，两条线从此独立演化。Fufan-CC 的 History Modal 里"继续会话"按钮走的正是 `resume` 路径，对应后端 `claudeAgentService.ts:101` 的 `resume: options.sessionId || undefined` 字段注入。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140059148.png" width=60%></div>

**Checkpoint 与文件回滚**

&emsp;&emsp;Checkpoint 让 AI 写过的文件改动可以一键回滚。`enable_file_checkpointing=True` 让 SDK 在每次 user turn 前自动对工作目录打文件快照，之后调用 `rewind_files(uuid)` 即可将文件系统还原到那个快照。

&emsp;&emsp;<font color=red>**这里有一个容易踩的隐藏依赖：必须同时配 `extra_args={"replay-user-messages": None}` 才能拿到正确的 `user_message_id`**</font>——SDK 0.2.87 `client.py:374` 的 `rewind_files` docstring 明示这一点。没设这个 flag 时，`UserMessage.uuid` 不是 CLI 端 checkpoint 索引用的真实 id，rewind 会报 `"No file checkpoint found for this message"`。下面代码段的 `ClaudeAgentOptions` 同时设了 `enable_file_checkpointing=True` 和 `extra_args={"replay-user-messages": None}`，两者缺一不可。

&emsp;&emsp;**另一个易踩的坑**：传给 AI 的 prompt 要用**绝对路径**指定文件位置（如 `用 Write 工具在 /tmp/xxx 创建...`）。`cwd` 参数只设了 CLI 子进程的工作目录，但 AI 自己解析相对路径时可能选别处（比如用户 `home` 目录），导致 cwd 内根本没文件被创建——checkpoint 自然就没东西可回滚。下面代码段用 `f"用 Write 工具在 {hello} 创建..."` 把绝对路径直接注入 prompt，避免这个陷阱。

&emsp;&emsp;<font color=red>这里有一个 API 形态切换：从 `query()` 单次调用模式，切到 `ClaudeSDKClient` 客户端模式</font>。两者的核心区别：**`query()` 是一次性流式生成器**，每次新建子进程跑完一轮就结束，适合"问一次答一次"的单轮交互；**`ClaudeSDKClient` 是持久客户端**，可以在同一个 client 实例上反复 `query()`、调 `rewind_files()`、做多轮交互，子进程持久驻留。WebSocket 长连接场景天然适合客户端模式——一个 WS 连接对应一个 `ClaudeSDKClient` 实例，比每条消息重启 CLI 子进程高效得多。Checkpoint 需要切换的原因是：`rewind_files` 是 `ClaudeSDKClient` 的实例方法，不属于 `query()` 函数，必须在持久 client 上调用。所以这里写成 `async with ClaudeSDKClient(...) as client:` 的形式。

In [9]:
import tempfile
from pathlib import Path
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, UserMessage, ResultMessage

# 关键：rewind_files 需要 extra_args={"replay-user-messages": None}
# 否则即使 enable_file_checkpointing=True，UserMessage.uuid 也不能用于 rewind
# （SDK 0.2.87 client.py rewind_files docstring 明示）
opts = ClaudeAgentOptions(
    cwd=tempfile.mkdtemp(prefix="sdk-rewind-"),
    permission_mode="bypassPermissions",
    enable_file_checkpointing=True,
    extra_args={"replay-user-messages": None},
    max_turns=5,
)

async with ClaudeSDKClient(options=opts) as client:
    # 阶段 A：让 AI 创建文件
    print("[A] 让 AI 创建文件...")
    user_uuid = None
    # 用绝对路径 prompt：cwd 参数只设了 CLI 子进程的工作目录，但 AI 自己解析
    # 相对路径时可能选别处（家目录），导致 cwd 内文件没真被创建 → checkpoint 失败
    hello = Path(opts.cwd) / "hello.txt"
    # SDK 0.2.87：client.query() 是"发送"协程，迭代要走 receive_response()
    await client.query(f"用 Write 工具在 {hello} 创建文件，内容只写 hello world")
    async for msg in client.receive_response():
        if isinstance(msg, UserMessage) and not user_uuid:
            user_uuid = getattr(msg, "uuid", None)
        elif isinstance(msg, ResultMessage):
            print(f"    cost=${(msg.total_cost_usd or 0):.4f}")

    print(f"[A] hello.txt 存在？{hello.exists()}  内容={hello.read_text().strip() if hello.exists() else 'N/A'}")

    # 阶段 B：回滚
    if user_uuid:
        print(f"\n[B] rewind to uuid={user_uuid[:8]}...")
        result = await client.rewind_files(user_uuid)
        print(f"[B] hello.txt 存在？{hello.exists()}（应为 False——已回滚）")

[A] 让 AI 创建文件...
    cost=$0.4240
[A] hello.txt 存在？True  内容=hello world

[B] rewind to uuid=8bb58afd...
[B] hello.txt 存在？False（应为 False——已回滚）


&emsp;&emsp;两个关键点需要区分清楚：① `rewind_files` 只回滚文件系统，不回滚对话历史——历史仍在，你可以继续对话，只是文件回到了快照状态；② Fufan-CC 在 stream 活跃时调 `stream.rewindFiles()` 走实时路径（TS 端方法名 camelCase），stream 结束后 fallback 到 `sessionManager.getSessionCheckpoints()` 从 JSONL 历史文件重建快照索引，两路逻辑在 `claudeAgentService.ts:180-193` 里做了分支判断。

> **【源码锚点】**：`claudeAgentService.ts:106`（`enableFileCheckpointing` 注入）、`claudeAgentService.ts:180-193`（`rewindFiles` 包装与 stream/fallback 分支）、`sessionManager.ts:727`（JSONL fallback 入口）

> **【踩坑预警】**：`rewind_files` 接收的 `uuid` 来自 `UserMessage.uuid`，不是 `session_id`。如果错传 `session_id`，SDK 会静默失败或抛 `InvalidCheckpointError`，文件不会回滚。务必在 `UserMessage` 事件里提取并保存 `uuid`。

---

### 4.6 WebSocket 桥与最小 Web 产品整合

&emsp;&emsp;前面 4 节我们都在 Python 脚本或 Jupyter cell 里直接跑 SDK。但 Fufan-CC 是一个 Web 产品——用户在浏览器里操作，服务器替用户调 SDK，再把结果实时推回浏览器。从"Python 脚本调 SDK"到"浏览器能用的 Web 服务"，这一跳的核心技术选型是 WebSocket：实时流式数据 + HIL 双向决策。

&emsp;&emsp;我们先来看这个选型背后的技术理由：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4.6-1 HTTP vs WebSocket：为什么 HIL 必须用 WebSocket</font></p>
<div class="center">

| 维度 | HTTP 请求-响应 | WebSocket 全双工 |
|:----:|:----:|:----:|
| 通信方向 | 一来一回，client 发 server 应 | 双向，任意一方随时可发 |
| 流式输出 | 需借助 SSE 或长轮询变通实现 | 天然支持，逐 frame 推送 |
| HIL 双向桥 | 无法做到（server 无法主动推弹窗给 client） | server 推 permission 弹窗，client 推用户决策回来 |

</div>

&emsp;&emsp;下面我们用最少的代码把这个桥搭出来。server 端负责接收 prompt、调 SDK、把消息流序列化成 JSON 推给 WebSocket；client 端负责连接、发 prompt、逐条收消息打印。

In [11]:
# WebSocket 桥：SDK 消息 → JSON → 浏览器
# 需要先安装：pip install websockets>=12.0
import json
import websockets
from claude_agent_sdk import (
    query, ClaudeAgentOptions,
    SystemMessage, AssistantMessage, TextBlock, ResultMessage,
)

async def sdk_handler(websocket):
    """每个 client 连接进来都跑一次 query，挑核心 3 类消息推回"""
    prompt = await websocket.recv()
    print(f"  [server] 收到 prompt: {prompt[:50]}")

    opts = ClaudeAgentOptions(max_turns=1, permission_mode="bypassPermissions")
    # 每条 SDK 消息按类型挑选关键字段序列化成 JSON 后推给浏览器
    async for msg in query(prompt=prompt, options=opts):
        if isinstance(msg, SystemMessage) and msg.subtype == "init":
            await websocket.send(json.dumps({
                "type": "init", "sid": msg.data.get("session_id")
            }))
        elif isinstance(msg, AssistantMessage):
            # 把 TextBlock 文本拆成独立 WS 帧，方便前端按 token 边到边渲染
            for block in msg.content:
                if isinstance(block, TextBlock):
                    await websocket.send(json.dumps({
                        "type": "text", "content": block.text
                    }))
        elif isinstance(msg, ResultMessage):
            # done 帧含本轮总成本，前端收到这条就停止 loading 状态
            await websocket.send(json.dumps({
                "type": "done", "cost": msg.total_cost_usd
            }))

ws_server = await websockets.serve(sdk_handler, "127.0.0.1", 8769)
print("[server] 监听 ws://127.0.0.1:8769")

[server] 监听 ws://127.0.0.1:8765


  [server] 收到 prompt: 用 5 个中文字告诉我什么是 LLM


&emsp;&emsp;server 跑起来之后，client 端只需要连接、发 prompt、循环收消息直到遇到 `"done"` 类型：

In [12]:
import json
import websockets

async def run_client():
    """连接 server，发 prompt，逐条收消息打印"""
    # async with 确保 WebSocket 连接在退出代码块时自动关闭，避免资源泄漏
    async with websockets.connect("ws://127.0.0.1:8769") as ws:
        await ws.send("用 5 个中文字告诉我什么是 LLM")
        # 循环收消息直到 server 推回 type=done 的收尾事件
        while True:
            msg = json.loads(await ws.recv())
            print(f"  [client] {msg}")
            if msg.get("type") == "done":
                break

await run_client()

  [client] {'type': 'init', 'sid': '6dcb7cbe-5827-4ef1-81f5-9383396919cb'}
  [client] {'type': 'text', 'content': '大语言模型'}
  [client] {'type': 'done', 'cost': 0.309125}


&emsp;&emsp;这两段代码就是 Fufan-CC `chatHandler.ts` 的核心骨架——server 端调 SDK 消费流，每条消息序列化成 JSON 推给 WebSocket client。真实项目在这个骨架上还叠加了：`EventEmitter` 分发（同一条 SDK 消息可能被多个下游订阅者消费）、`AUTO_APPROVE` 工具白名单（低风险工具跳过 permission 弹窗）、HIL 双向桥（server 主动推 permission 请求，等 client 推回用户决策）、心跳检测与连接清理。但它们都是在我们这个两段代码的骨架上加的策略层，骨架本身没有变。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140104425.png" width=60%></div>

&emsp;&emsp;<font color=red>到这里你已经走完了 Fufan-CC 后端的完整核心链路——从 SDK 入口（4.2）、消息归一化（4.3）、HIL 异步桥接（4.4）、Session 状态管理（4.5），到 WebSocket 双向通信（4.6）。这 5 个模块拼在一起，就是 `Fufan-CC` 后端约 1 万行 TypeScript 的 Python 精简版（约 200 行 Python 对应原项目 5% 的关键路径）。</font>

&emsp;&emsp;当然，精简版和完整版之间还有相当的距离。下面这张表把两者的主要差距一次性列清楚，让你对后续阅读 TypeScript 源码有合理的心理预期：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4.6-2 课件 MVP vs Fufan-CC 完整版</font></p>
<div class="center">

| 维度 | 课件 MVP（本章） | Fufan-CC 完整版 |
|:----:|:----:|:----:|
| 代码量 | 约 200 行 Python（本课精简版） | 约 1 万行 TypeScript（含前后端完整工程） |
| 消息类型 | 5 种核心事件 | 10+ 种（含 `compact_boundary` / `context_usage` 等） |
| Session 管理 | 最小状态（内存） | 持久化、列表、切换、历史 Modal |
| 工具集成 | SDK 内置 80 个 | + MCP / Skill / Plugin / Hook |
| 后端治理 | 最小异常处理 | abort、心跳、超时、日志、连接清理 |

</div>

> **【源码锚点】**：`server/src/websocket/chatHandler.ts`（WebSocket 路由与消息分发）、`server/src/services/claudeAgentService.ts`（SDK 封装与 EventEmitter 集成）

---

## <center>第五章：设计哲学 + 迁移启示</center>

&emsp;&emsp;前四章从本地部署、功能走读、架构拆解到 SDK Python 复现，完成了"能跑 + 能看懂 + 能动手"这条主线。最后一章不再加代码，而是回答一个更上层的问题：**这个项目为什么这样设计？** 核心概念是"零侵入哲学"——`Fufan-CC` 不发明自己的数据存储，而是直接读 CLI 落盘的事实源（`~/.claude/projects/<hash>/*.jsonl`），自己只做"读取 + 加工 + 渲染"。理解这个选择的背后逻辑，比记住任何一段代码都更有迁移价值。

&emsp;&emsp;本章三个小节依次递进：先用真跑代码让你亲眼看清"事实源"和 4 件加工的对应关系（5.1），再把整门课的核心范式抽象成可以套到任意 CLI 的 5 步迁移骨架（5.2），最后用全课速查表、源码入口表、能力边界表和分层练习收束（5.3）。

### 5.1 零侵入哲学：事实源与 4 件加工

&emsp;&emsp;零侵入的核心是"自己不发明数据存储，但必须发明数据解释层"。`Claude Code CLI` 把每一次对话落盘到 `~/.claude/projects/<hash>/*.jsonl`，`Fufan-CC` 对这批数据只做"读取 + 加工 + 渲染"，写入完全交给 CLI。我们先用一段真跑代码看 CLI 自己落盘的 JSONL 长什么样：

In [1]:
# 零侵入读取原始 JSONL — 看 CLI 自己落盘的事实源
# 目的：让你亲眼看到 Fufan-CC 不存数据库，CLI 已经把所有对话落盘了

from pathlib import Path
import json

# Claude Code CLI 的默认数据目录
projects_dir = Path.home() / ".claude" / "projects"

# 列出所有项目目录（每个项目是一个被 pathToHash 编码的子目录）
if not projects_dir.exists():
    print(f"目录不存在：{projects_dir}")
    print("请先运行一次 Claude Code CLI 让它创建该目录")
else:
    project_dirs = sorted(projects_dir.iterdir())
    print(f"发现 {len(project_dirs)} 个项目目录（前 3 个）:")
    for d in project_dirs[:3]:
        print(f"  - {d.name}")

    # 找一个有 session 文件的项目，打印第一个 session 的前 3 条记录
    for proj_dir in project_dirs:
        # *.jsonl 是 session transcript，sessions-index.json 是元数据索引
        jsonl_files = sorted(proj_dir.glob("*.jsonl"))
        if not jsonl_files:
            continue

        first_session = jsonl_files[0]
        print(f"\n读取 session: {proj_dir.name}/{first_session.name}")
        # JSONL 是"一行一条 JSON"格式，每行单独 parse
        with first_session.open("r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 3:
                    break
                entry = json.loads(line)
                # 只打印关键字段避免输出过长
                print(f"  [{i}] type={entry.get('type')!r}, "
                      f"uuid={entry.get('uuid', '')[:8]!r}, "
                      f"timestamp={entry.get('timestamp', '')[:19]!r}")
        break  # 演示一个就够

发现 77 个项目目录（前 3 个）:
  - -Users-mac
  - -Users-mac--------------
  - -Users-mac----------------

读取 session: -Users-mac/010eac7a-d2fa-4867-ac35-4d11df7acc60.jsonl
  [0] type='permission-mode', uuid='', timestamp=''
  [1] type='file-history-snapshot', uuid='', timestamp=''
  [2] type='user', uuid='420c0fc2', timestamp='2026-05-23T13:08:20'


&emsp;&emsp;运行后你会看到三类记录：`type="user"` 是用户发的消息、`type="assistant"` 是 Claude 的回复、`type="file-history-snapshot"` 是 CLI 自己记录的文件版本快照（用于支持回滚）。还有 `type="summary"` / `type="system"` 等更多类型，往后翻 session 文件能看到。**这就是"事实源"——`Fufan-CC` 完全不存数据库，所有对话历史 / 工具调用 / 文件快照都在这里**。

&emsp;&emsp;原始 JSONL 不能直接喂给 UI——UI 需要会话列表、消息过滤、checkpoint 时间线等结构化数据。这些信息在原始 JSONL 里都有，但散落在不同位置、有不同的编码方式、还混杂着 CLI 自己注入的"内部消息"。这就是 `sessionManager.ts` 这个 970 行文件存在的原因：它是加工层的核心，做完 4 件事后才能把事实源翻译成 UI 能用的结构。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Fufan-CC sessionManager 4 件加工</font></p>
<div class="center">

| 加工项 | 原始输入 | Fufan-CC 产出 | 为什么 UI 需要 | 源码锚点 |
|:------:|:--------:|:-------------:|:--------------:|:--------:|
| 路径哈希编码 | 用户输入的项目路径 | `~/.claude/projects/<hash>/` 目录定位 | UI 要知道去哪个目录读 session | `sessionManager.ts:19` |
| sessions-index 解析 | `sessions-index.json` 原始 JSON | 会话列表（summary / model / branch / count） | 渲染 HistoryModal 不需要 parse 全部 JSONL | `sessionManager.ts:156-189` |
| 内部消息过滤 | JSONL 中所有 user 消息 | 过滤掉 `[` / `<` / "This session" 开头的内部消息 | 防止 UI 显示 CLI 注入的系统指令 | `sessionManager.ts:30-34` |
| checkpoint 重建 | JSONL 中 `file-history-snapshot` 记录 | checkpoint 时间线（messageId → changedFiles） | 渲染左栏 Checkpoint 时间线 | `sessionManager.ts:727-820` |

</div>

&emsp;&emsp;下面这段探针代码用约 30 行 Python 验证 4 件加工各自的输入——路径哈希能算、索引文件能读、内部消息能识别、checkpoint 记录能统计：

In [13]:
# 4 件加工探针：读取一个 session 目录，输出加工层需要处理的 4 类数据统计
from pathlib import Path
import json, re

def path_to_hash(p: str) -> str:
    """加工 1：路径哈希编码（对应 sessionManager.ts:19）"""
    return re.sub(r"[^a-zA-Z0-9]", "-", p.replace("\\", "/"))

def is_internal_message(text: str) -> bool:
    """加工 3：内部消息过滤（对应 sessionManager.ts:30-34）"""
    return text.startswith("[") or text.startswith("<") or text.startswith("This session is being continued")

def _extract_user_text(entry: dict) -> str:
    """从 user 条目中提取首段 text。
    JSONL 中 message.content 既可能是 str（CLI 较新版本），也可能是 list[dict]（含 ToolResult 的会话）。
    需要类型守卫，否则 list[0] 是 dict 但循环里第一条 user 消息恰好 content=str 时会 AttributeError。
    """
    msg = entry.get("message", {})
    if not isinstance(msg, dict):
        return ""
    content = msg.get("content", "")
    if isinstance(content, str):
        return content
    if isinstance(content, list) and content and isinstance(content[0], dict):
        return content[0].get("text", "")
    return ""

# 探测一个真实 session
projects_dir = Path.home() / ".claude" / "projects"
for proj_dir in sorted(projects_dir.iterdir()):
    jsonl_files = sorted(proj_dir.glob("*.jsonl"))
    index_file = proj_dir / "sessions-index.json"
    if not jsonl_files:
        continue

    print(f"项目目录: {proj_dir.name}")
    print(f"  加工 1 · 路径哈希示例: path_to_hash('/Users/mac/proj') → {path_to_hash('/Users/mac/proj')!r}")
    print(f"  加工 2 · sessions-index.json 存在？{'是' if index_file.exists() else '否'}")

    # 统计一个 session 的消息类型分布
    with jsonl_files[0].open("r") as f:
        entries = [json.loads(line) for line in f]
    types = {}
    internal_count = 0
    for e in entries:
        t = e.get("type", "unknown")
        types[t] = types.get(t, 0) + 1
        if t == "user" and is_internal_message(_extract_user_text(e)):
            internal_count += 1

    print(f"  加工 3 · 消息类型分布: {dict(sorted(types.items()))}")
    print(f"  加工 3 · 内部消息数: {internal_count}")
    print(f"  加工 4 · checkpoint 记录数: {types.get('file-history-snapshot', 0)}")
    break

项目目录: -Users-mac
  加工 1 · 路径哈希示例: path_to_hash('/Users/mac/proj') → '-Users-mac-proj'
  加工 2 · sessions-index.json 存在？否
  加工 3 · 消息类型分布: {'agent-name': 2, 'ai-title': 147, 'assistant': 969, 'attachment': 84, 'custom-title': 2, 'file-history-snapshot': 97, 'last-prompt': 147, 'permission-mode': 148, 'queue-operation': 58, 'system': 75, 'user': 659}
  加工 3 · 内部消息数: 63
  加工 4 · checkpoint 记录数: 97


&emsp;&emsp;这个探针用 30 行代码验证了 4 件加工的输入——`sessionManager.ts` 的 970 行代码，核心就是把这 4 件事做完整、做健壮：处理边界情况、合并增量 snapshot、关联 messageId 与用户消息内容、过滤各种格式的内部消息。探针做的是"能统计"，生产代码做的是"无遗漏地翻译"。

&emsp;&emsp;**零侵入有三个不可替代的好处**：第一，没有双写不一致——CLI 写 JSONL 你不写 DB，不存在中途断电导致两边对不上的 bug。第二，用户随时可以"逃离"——学员今天用 Web UI，明天回到纯 CLI，历史数据原封不动。第三，CLI 升级不会破坏你——只要 JSONL 格式不变，`Fufan-CC` 不用改一行代码。**这三个好处是零侵入路线独有的，隐含约束是：CLI 必须本身落盘且落盘格式相对稳定**。如果目标 CLI 完全不落盘，或落盘格式频繁破坏性变更，需要退化为"自建数据库 + 定期回填"的混合模式。

> **【源码锚点】**：`server/src/services/sessionManager.ts:1-970`（JSONL 读取 + 4 件加工 + checkpoint 重建全在这个文件）

### 5.2 迁移到任意 CLI 的 5 步骨架

&emsp;&emsp;把整门课抽象成方法论：如果有人让你把 `gh CLI` / `aider` / `goose` 包成 Web 应用，按这 5 步走。每步的关键不是操作本身，而是**在那个步骤必须做的那个决策**。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>CLI 集成进 Web 的 5 步迁移骨架</font></p>
<div class="center">

| 步骤 | 做什么 | 本课对应 | 关键决策 |
|:----:|:------:|:--------:|:--------:|
| 1. PTY 起进程 | 用 `node-pty` / `pexpect` spawn CLI | `ptyService.ts` | 先验证"能在应用进程里双向通 stdin/stdout" |
| 2. SDK/spawn 接入 | 优先用官方 SDK，没有则 PTY 自行解析 | `claudeAgentService.ts` | 判断要不要双泳道（Agent + Terminal 并存？） |
| 3. HIL 异步桥接 | async handler + Future/Promise + WebSocket | `requestPermission()` | 超时 / 并发 / 取消 / 白名单 4 点优先排查 |
| 4. 状态共享 | 优先零侵入读 CLI 落盘；CLI 不落盘则自建 | `sessionManager.ts` | 先问"CLI 有没有稳定的落盘格式？" |
| 5. 边界判断 | 应用层只做"翻译器"，不重新发明 CLI 已有能力 | 全课设计哲学 | 任何"在应用层再加一层智能"的冲动都先质疑 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260528140033210.png" width=60%></div>

&emsp;&emsp;（5 步迁移骨架决策路径——每步标注"关键决策问题"，失败路径用虚线回退，让你把这张图作为随身 checklist 记住）

&emsp;&emsp;**第 1 步**验证的是可行性地基——只要 PTY 起不来，后面四步都白搭。`ptyService.ts` 做的事就是把 `node-pty` 的 spawn / write / onData 三个原语包成可复用的服务。**第 2 步**的核心决策是"单泳道还是双泳道"——如果目标 CLI 既要 Agent 对话又要用户直接敲命令，就必须像 `Fufan-CC` 一样开两条独立进程链；只需要单一能力则单泳道更简单。**第 3 步**对应 4.4 节的 7 维 checklist，超时、并发、取消、白名单这 4 点最容易踩坑，优先设计。**第 4 步**是最大的设计选择题，先走零侵入，实在不行再自建。**第 5 步**不是一个操作，而是一个持续的纪律——应用层定位是"翻译器"，任何想"在应用层再加一层智能"的冲动（自己再调一次模型做前置规划、自己维护一份知识库）都应该先问：这不是 CLI 自己已经能做的吗？

&emsp;&emsp;**课后挑战：maxBudget 控件**

&emsp;&emsp;给 `Fufan-CC` 加一个"本次对话最大 token 预算"数字输入框：用户填入预算值，超过阈值时后端主动停止流式输出。动手前先用 4 类后端转译做预判——SDK 在 `ClaudeAgentOptions` 内置了 `max_budget_usd: float | None` 字段，超出后会自动终止流。如果只需要简单的预算上限（不带 UI 反馈），直接 `ClaudeAgentOptions(max_budget_usd=0.05)` 就够了，属于**纯 SDK**；如果你要做的是细粒度的 token 进度条可视化（实时更新 UI），才需要在 `consumeStream` 消费每条 message 时累加 `usage.input_tokens + usage.output_tokens`，超限后调 `abort()`——这部分属于**后端加工**。接入链路参考 `runMode` 的做法：前端 `InputBar` payload 加 `maxBudget` 字段 → `chatHandler.ts` 解析 → 传入 `ClaudeAgentService.start()`，这条链路接通才算"按钮真生效"。难度：中等 / 预计耗时：1-2 小时 / 验收标准：UI 填预算 → 超限自动停止 → token 进度条变红。

### 5.3 全课总结与课后路线

&emsp;&emsp;用两张速查表 + 一张能力边界表 + 分层练习收束全课。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>全课 8 个核心概念速查</font></p>
<div class="center">

| 概念 | 一句话定义 | 首次出现 |
|:----:|:----------:|:--------:|
| 双泳道架构 | Chat/Agent 泳道走 SDK，Terminal 泳道走 `node-pty`，互不感知 | 3.2 节 |
| 5 层分层 | 浏览器 → WebSocket → Node 服务 → CLI 进程 → 本地文件系统 | 3.1 节 |
| 8 步飞行路径 | 一次用户输入在 5 层间走完的完整时序 | 3.3 节 |
| 4 类后端转译 | prompt 改写 / 后端调度 / 后端加工 / UI 聚合 | 4.1 节 |
| HIL 异步桥接 | CLI 暂停（await Future）→ WebSocket 推弹窗 → 用户决策回传 | 4.4 节 |
| 事件归一化 | `SDKMessage` → 稳定的产品事件协议，隔离 SDK 变更 | 4.3 节 |
| 零侵入哲学 | 不发明数据存储，直接读 CLI 落盘的事实源 | 5.1 节 |
| 5 步迁移骨架 | PTY → SDK → HIL → 状态 → 边界，可套到任意 CLI | 5.2 节 |

</div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>课后读源码入口</font></p>
<div class="center">

| 文件 | 行数 | 阅读目标 |
|:----:|:----:|:--------:|
| `chatHandler.ts` | 211 | WebSocket 路由 + HIL 白名单转发 |
| `claudeAgentService.ts` | 607 | SDK 封装 + query 流消费 + HIL Promise 桥接 |
| `sessionManager.ts` | 970 | JSONL 读取 + 4 件加工 + checkpoint 重建 |
| `ptyService.ts` | 约 110 | 终端泳道 `node-pty` 封装 |
| `AppLayout.tsx` | ~50 | 前端 3 栏布局入口 |
| `ChatPanel.tsx` | ~300 | 对话 UI + 工具卡片 + HIL 弹窗 |

</div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>学完本课能做什么 vs 不能做什么</font></p>
<div class="center">

| 能做（基于本课学到的认知）| 不能做（需要本课没覆盖的能力）|
|:-------------------------:|:------------------------------:|
| 在自己机器上完整部署 Fufan-CC 并跑通全部功能域 | 给 Fufan-CC 提交一个生产级 bugfix（需要熟悉完整测试链 + CI/CD） |
| 用一张图给同事讲清双泳道架构 + 5 层 + 8 步飞行路径 | 独立设计高可用多实例部署方案（需要 Nginx 反代 + PM2 + Docker 编排） |
| 用 4 类后端转译对任意 CLI 集成进 Web 的项目做架构归纳 | 给 Fufan-CC 提交新功能 PR（需熟悉 Zustand + Tailwind CSS v4 + 测试链） |
| 用 Python `claude-agent-sdk` 跑通基础对话 + HIL 异步桥接最小版 | 独立完成"包任意 CLI 的完整工程方案"（5 步骨架是骨架，每步往深还有大量细节） |
| 讲清"零侵入读 `~/.claude/projects/`"是什么、为什么、4 件加工各是什么 | 在本机调试 Fufan-CC 任意 bug（需熟悉 `pnpm dev` 启动链、`ws` 调试、React DevTools） |

</div>

&emsp;&emsp;**分层课后练习**

&emsp;&emsp;**15 分钟 · 阅读题**：打开 `chatHandler.ts`，找到 `AUTO_APPROVE_TOOLS` 白名单，回答：如果把 `Edit` 加进白名单会怎样？`Fufan-CC` 的 HIL 弹窗还会弹出来吗？

&emsp;&emsp;**45 分钟 · 小改造**：给 Fufan-CC 的"拓展 → Memory"面板加一个"导出为 Markdown"按钮——前端读 `memoryService` 的数据，格式化输出到剪贴板或下载文件。

&emsp;&emsp;**2 小时 · 扩展题**：完成 5.2 节的 maxBudget 控件——UI 填预算 → 超限自动停止 → token 进度条变红。

&emsp;&emsp;如果还有疑问，建议按三个方向去问：**架构选择类**——"我手上有个 CLI 想包成 Web，单泳道还是双泳道？哪些情况必须双泳道？"；**HIL 工程化类**——"生产版的 7 个差异点哪个最容易踩坑？怎么排查？"；**零侵入边界类**——"如果 CLI 不落盘，除了自建数据库还有别的路吗？"这三个方向是接下来最值得深挖的，每个往下都能开一门完整的深度课。

&emsp;&emsp;<font color=red>整门课的核心判断习惯：下次看任何 CLI 集成进 Web 的项目，先问三个问题——"这个 UI 交互背后是 SDK 给的还是后端转译的？数据落在哪？谁是事实源？"</font>

&emsp;&emsp;**最后回头看一眼这趟旅程**——课前你看到的是一个神秘的 Web 聊天框，工具调用、HIL 弹窗、会话切换、Checkpoint 回滚这些功能你只能"用"，但说不清后端到底做了什么；课后你能读懂它背后的 5 层架构、画得出 8 步飞行路径、知道每个 UI 控件对应哪个 SDK 参数、能用约 200 行 Python 复现 HIL 异步桥接和 WebSocket 整合的核心机制、还能讲清"零侵入读 JSONL 事实源"这个设计选择的代价和好处。这是从**使用者**到**架构理解者**的跨越——你已经具备了把任意 CLI 工具包成 Web 应用的判断框架。